# Pipeline Korpus Multi-genre Batak Toba dan Indonesia (v2)

Notebook ini menjalankan **seluruh pengolahan korpus multi-genre dalam satu tempat**, mulai dari hasil kerja kedua anotator (`Combined Annotator.xlsx`) sampai korpus paralel tingkat kalimat yang siap diuji kesepakatan antaranotator (IAA).

Penelitian: *Digitalisasi dan Praproses Korpus Bahasa Batak Toba sebagai Fondasi Pengembangan Model Bahasa Lokal Berbasis Deep Learning* (Skema PDP, Universitas Pelita Harapan).

## Prinsip yang dipegang notebook ini

1. **Satu sumber kebenaran.** Titik awal hanya satu berkas, yaitu `Combined Annotator.xlsx`. Semua berkas lain adalah turunan yang bisa dibuat ulang.
2. **Semua perubahan tercatat.** Tidak ada perbaikan manual di luar notebook. Perbaikan label, pembersihan teks, dan keputusan pemrosesan ditulis sebagai kode dan dicatat di berkas log.
3. **Parameter ditetapkan sebelum dijalankan.** Semua ambang dan aturan ada di sel **Konfigurasi**. Jangan mengubah parameter setelah melihat hasil hanya agar angkanya terlihat lebih baik. Kalau parameter memang perlu diubah, catat alasannya.
4. **Titik simpan (checkpoint).** Setiap tahap menyimpan keluarannya ke Google Drive. Saat notebook dijalankan ulang, tahap yang masukan dan parameternya tidak berubah akan diambil dari checkpoint, tidak dihitung ulang.
5. **Bisa diulang.** Seed acak tetap, versi pustaka dicatat, dan sidik SHA-256 setiap berkas masukan dan keluaran disimpan di manifest.

## Cakupan

| Termasuk | Tidak termasuk |
|---|---|
| Sub-korpus multi-genre dari dua anotator (termasuk Agenda HKBP) | Sub-korpus Alkitab (diolah di notebook terpisah) |
| Statistik korpus dengan tiga metode hitung token | Pelatihan model atau eksperimen downstream |
| Korpus paralel tingkat kalimat (utama dan liturgi) | Penerapan hasil adjudikasi ke korpus (tahap berikutnya setelah IAA) |
| Lembar review untuk dua anotator dan penghitungan IAA | Anotasi Universal Dependencies |

## Peta tahapan

| Tahap | Nama | Apa yang dikerjakan | Keluaran utama | Dipakai untuk |
|---|---|---|---|---|
| 0 | Persiapan | Memasang pustaka, menghubungkan Google Drive, membaca konfigurasi | Struktur folder kerja | Semua tahap |
| 1 | Muat dan validasi input | Membaca `Combined Annotator.xlsx`, memberi `doc_id` tetap, memeriksa kelengkapan | `s01_dokumen_mentah.csv`, `s01_validasi.json` | Tahap 2 |
| 2 | Koreksi metadata | Menyeragamkan nama kolom, memperbaiki label yang salah, menandai sumber kosong | `s02_dokumen.csv`, `s02_log_koreksi.csv` | Tahap 3 |
| 3 | Pembersihan ringan | Menormalkan Unicode dan spasi tanpa mengubah kata | `s03_btb_multigenre.csv` (berkas rilis), `s03_log_pembersihan.csv` | Tahap 4, 5, rilis |
| 4 | Segmentasi kalimat | Memecah setiap dokumen menjadi kalimat atau baris | `s04_kalimat.csv`, `s04_ringkasan_segmentasi.csv` | Tahap 5, 6 |
| 5 | Statistik korpus | Menghitung dokumen, kalimat, dan token dengan tiga metode | `s05_statistik.json`, tabel per genre, potongan README | README, laporan teknis, paper |
| 6 | Penyelarasan kalimat | Memasangkan kalimat Toba dan Indonesia (aturan 1:1 atau LaBSE) | `s06_penyelarasan.csv`, `s06_ringkasan_penyelarasan.csv` | Tahap 7 |
| 7 | Audit otomatis | Memberi tanda masalah, prioritas review, dan status pada setiap pasangan | `s07_paralel_utama.csv`, `s07_paralel_liturgi.csv`, `s07_tak_berpasangan.csv` | Tahap 8, rilis |
| 8 | Lembar review IAA | Menyusun sampel dan dua lembar kerja identik untuk dua anotator | `s08_review_anotator_A.xlsx`, `s08_review_anotator_B.xlsx`, `s08_kunci_sampel.csv` | Diisi anotator, lalu tahap 9 |
| 9 | IAA dan adjudikasi | Menghitung Cohen's kappa dan menyusun lembar adjudikasi | `s09_laporan_iaa.json`, `s09_adjudikasi.xlsx` | Paper, validasi v3 |
| 10 | Paket rilis | Menyalin berkas final, membandingkan dengan v1, menulis manifest dan changelog | Folder `rilis/`, `manifest.json`, `CHANGELOG_v2.md` | Repositori GitHub, paper |

## Cara menjalankan

1. Buat folder kerja di Google Drive (bawaan: `MyDrive/Datasets/btb-pipeline-v2`), lalu buat subfolder `input`.
2. Letakkan `Combined Annotator.xlsx` di `input/`. Opsional: unduh `data/experimental/btb_parallel_sentence_v1.csv` dari repositori GitHub dan letakkan juga di `input/` agar tahap 10 bisa membandingkan hasil v2 dengan v1.
3. Pilih **Runtime > Change runtime type > T4 GPU**. GPU hanya dibutuhkan tahap 6, tetapi tanpa GPU tahap itu jauh lebih lambat.
4. Jalankan **Runtime > Run all**.
5. Bagikan `s08_review_anotator_A.xlsx` dan `s08_review_anotator_B.xlsx` kepada dua anotator. Mereka bekerja **terpisah dan tidak saling melihat**.
6. Setelah selesai, simpan kedua berkas yang sudah diisi ke subfolder `review_terisi/` dengan nama `review_anotator_A.xlsx` dan `review_anotator_B.xlsx`, lalu jalankan **Run all** sekali lagi. Tahap 1 sampai 8 akan diambil dari checkpoint, dan tahap 9 akan menghitung IAA.

**Penting:** jangan menjalankan sel secara acak atau melompat-lompat. Notebook ini dirancang untuk dijalankan dari atas ke bawah. Angka "520 dokumen" pada EDA versi lama muncul karena sel dijalankan tidak berurutan.

---
# Tahap 0. Persiapan

**Tujuan:** menyiapkan lingkungan kerja yang sama setiap kali notebook dijalankan.

**Yang dikerjakan:**
1. Memasang pustaka yang dibutuhkan (`sentence-transformers` untuk LaBSE, `openpyxl` untuk berkas Excel, `nltk` untuk pembanding hitungan token versi lama).
2. Menghubungkan Google Drive bila notebook berjalan di Colab.
3. Membaca konfigurasi, menyiapkan struktur folder kerja, dan mendefinisikan fungsi bantu checkpoint.

**Keluaran yang diharapkan:** folder `input/`, `keluaran/`, `keluaran/_meta/`, `review_terisi/`, dan `rilis/` di dalam folder kerja.

**Dipakai untuk:** semua tahap berikutnya.

In [ ]:
%pip install -q "sentence-transformers>=2.7" openpyxl nltk

In [ ]:
import os
import sys
import json
import math
import random
import shutil
import hashlib
import platform
import unicodedata
import re
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from google.colab import drive
    DI_COLAB = True
except ImportError:
    DI_COLAB = False

if DI_COLAB:
    drive.mount("/content/drive")

pd.set_option("display.max_colwidth", 80)
print("Berjalan di Colab:", DI_COLAB)
print("Python", platform.python_version(), "| pandas", pd.__version__)

## Konfigurasi

Seluruh parameter pipeline ada di satu sel ini. Setiap tahap mencatat sidik (hash) dari parameter yang dipakainya. Bila parameter sebuah tahap berubah, tahap itu dan semua tahap sesudahnya otomatis dihitung ulang saat **Run all**.

Penjelasan bagian konfigurasi:

| Bagian | Isi | Dipakai di tahap |
|---|---|---|
| `folder_kerja` | Lokasi folder kerja di Google Drive | Semua |
| `paksa_ulang` | Daftar kode tahap yang dipaksa dihitung ulang, misalnya `["s06"]` | Semua |
| `kolom_anotator` | Pemetaan nama kolom Google Sheet anotator ke nama kolom baku | 1, 2 |
| `koreksi_metadata` | Daftar perbaikan label yang disengaja, lengkap dengan alasannya | 2 |
| `normalisasi_label` | Penyeragaman penulisan label genre dan sub-genre | 2 |
| `segmentasi` | Aturan pemecahan kalimat | 4 |
| `penyelarasan` | Model, strategi per sub-genre, dan penalti algoritme | 6 |
| `audit` | Ambang untuk tanda masalah otomatis | 7 |
| `review` | Desain sampel dan kategori keputusan anotator | 8, 9 |

In [ ]:
KONFIGURASI = {
    "versi_pipeline": "2.0.0",
    "folder_kerja": os.environ.get("BTB_FOLDER_KERJA", "/content/drive/MyDrive/Datasets/btb-pipeline-v2"),
    "berkas_anotator": "Combined Annotator.xlsx",
    "sheet_anotator": 0,
    "berkas_pembanding_v1": "btb_parallel_sentence_v1.csv",
    "seed": 42,
    "paksa_ulang": [],
    "target_token": 100_000,

    "kolom_anotator": {
        "Column 1": "nomor_anotator",
        "Date Collected": "date_collected",
        "Title": "title",
        "Text in Toba": "text_bt",
        "Text in Bahasa Indonesia": "text_id",
        "Genre Category": "genre",
        "Sub-Genre Category": "subgenre",
        "Original Author (Publisher or Person)": "original_author",
        "Source Link": "source_link",
        "Additional Notes (optional)": "notes",
    },
    "kolom_wajib": ["Title", "Text in Toba", "Text in Bahasa Indonesia", "Genre Category", "Sub-Genre Category"],

    "koreksi_metadata": [
        {
            "judul_mengandung": ["Jangan Membuat Masalah Kecil", "Ikigai"],
            "kolom": "genre",
            "nilai_baru": "Educational",
            "alasan": "Ulasan buku, tercantum di daftar sumber Educational (Book summary) pada panduan anotator. Label lama berasal dari isian contoh templat.",
        },
        {
            "judul_mengandung": ["Kota Cerdas Berkelanjutan", "Jangan Membuat Masalah Kecil", "Ikigai"],
            "kolom": "subgenre",
            "nilai_baru": "Book Summary",
            "alasan": "Ulasan buku, tercantum di daftar sumber Educational (Book summary) pada panduan anotator. Label lama berasal dari isian contoh templat.",
        },
        {
            "judul_mengandung": [
                "Analisis Literasi Artifical Intelligence",
                "Implementasi Teknologi Artificial Intelligence",
                "Metafora cinta dalam bahasa Batak Toba",
                "ANALISIS PENERJEMAHAN SEJARAH DAN BUDAYA",
                "Peran Usaha Mikro, Kecil Dan Menengah",
                "Dampak Ekonomi Digital Terhadap Pertumbuhan",
                "SIKAP ETIS UMAT KRISTEN",
                "Pendidik sebagai Pembimbing Rohani",
                "Kajian Doa Menurut Alkitab",
            ],
            "kolom": "subgenre",
            "nilai_baru": "Abstract",
            "alasan": "Abstrak artikel jurnal, tercantum di daftar sumber Educational (Abstract) pada panduan anotator. Label lama News Article tidak sesuai.",
        },
        {
            "judul_mengandung": ["Raja Martua Doli"],
            "kolom": "source_link",
            "nilai_baru": "https://www.hutapea.id/2019/06/torsa-torsa-raja-martua-doli-dohot.html",
            "alasan": "Kolom sumber berisi penanda [URL]. Tautan diambil dari daftar sumber Annotator #1 pada notebook panduan anotator.",
        },
    ],
    "isian_templat": {
        "original_author": ["Unknown", "Community", "Dr. Sari Dewi", "Local Council", "Tempo Magazine"],
        "notes": [
            "Needs cross-referencing with other Batak tales.",
            "Translation in progress for key sections.",
            "Compare with older versions.",
            "Used for online course.",
            "Awaiting full transcription.",
            "Part of media analysis project.",
        ],
        "date_collected": [
            "2025-10-01 00:00:00", "2024-11-20 00:00:00", "2025-01-05 00:00:00",
            "2025-03-22 00:00:00", "2025-07-10 00:00:00", "2024-12-01 00:00:00",
        ],
    },
    "normalisasi_label": {
        "genre": {"Historical/ Traditional": "Historical/Traditional"},
        "subgenre": {},
    },
    "placeholder_sumber": ["[URL]", "URL", "-", "n/a"],

    "segmentasi": {
        "pisah_setiap_baris_baru": True,
        "singkatan": [
            "dr", "drs", "dra", "ir", "prof", "hj", "h", "tn", "ny", "sdr",
            "yth", "dll", "dst", "dsb", "tsb", "tgl", "no", "hlm",
            "jl", "jln", "kel", "kec", "kab", "prov",
            "st", "sd", "smp", "sma", "smk", "pt", "cv", "tbk",
            "op", "ompu", "amang", "inang", "ama", "ina",
            "vs", "etc", "vol",
        ],
    },

    "penyelarasan": {
        "model": "sentence-transformers/LaBSE",
        "ukuran_batch": 64,
        "strategi_per_subgenre": {
            "Prayers": "rule",
            "Peribahasa/Umpama": "rule",
        },
        "strategi_bawaan": "labse",
        "subgenre_liturgi": ["Ritual Text"],
        "tipe_blok": [[1, 1], [1, 2], [2, 1], [1, 0], [0, 1], [2, 2], [1, 3], [3, 1]],
        "penalti_null": 0.5,
        "penalti_gabung": 0.1,
    },

    "audit": {
        "rasio_karakter_min": 0.4,
        "rasio_karakter_maks": 2.5,
        "rasio_token_ekstrem": 2.0,
        "panjang_unit_maks": 45,
        "sim_rendah": 0.2,
        "fragmen_maks_token": 3,
        "tanda_prioritas_tinggi": ["tipe_bukan_1_1", "sisi_identik", "rasio_karakter_ekstrem"],
    },

    "review": {
        "proporsi_sampel_rutin": 0.10,
        "minimal_sampel_rutin": 100,
        "sertakan_liturgi": False,
        "kategori_keputusan": {
            "terima": "Pasangan sudah benar. Batas unit tepat di kedua sisi dan maknanya sepadan.",
            "perbaiki_penyelarasan": "Batas penyelarasan salah, tetapi teks kedua sisi benar. Misalnya sebagian kalimat Indonesia sebenarnya milik unit sebelum atau sesudahnya.",
            "perbaiki_terjemahan": "Batas unit sudah benar, tetapi terjemahan Indonesia keliru, kurang, atau berlebih.",
            "fragmen": "Pasangan sepadan, tetapi unitnya bukan kalimat lengkap. Contohnya judul, butir daftar, atau pengantar ujaran.",
            "perlu_sumber": "Belum bisa memutuskan tanpa melihat teks asli atau konteks yang lebih luas.",
            "keluarkan": "Pasangan harus dikeluarkan dari korpus. Contohnya kedua sisi tidak berpadanan sama sekali, atau kedua sisi berbahasa sama.",
        },
        "bootstrap_iaa": 1000,
    },
}

random.seed(KONFIGURASI["seed"])
np.random.seed(KONFIGURASI["seed"])

ROOT = Path(KONFIGURASI["folder_kerja"])
DIR_INPUT = ROOT / "input"
DIR_OUT = ROOT / "keluaran"
DIR_META = DIR_OUT / "_meta"
DIR_REVIEW = ROOT / "review_terisi"
DIR_RILIS = ROOT / "rilis"
for d in (DIR_INPUT, DIR_OUT, DIR_META, DIR_REVIEW, DIR_RILIS):
    d.mkdir(parents=True, exist_ok=True)

BERKAS_ANOTATOR = DIR_INPUT / KONFIGURASI["berkas_anotator"]
print("Folder kerja:", ROOT)
print("Berkas anotator:", BERKAS_ANOTATOR, "| ada:", BERKAS_ANOTATOR.exists())

## Fungsi bantu checkpoint

Setiap tahap dibungkus objek `Tahap`. Objek ini menghitung **sidik** dari empat hal: parameter tahap, isi berkas masukan (SHA-256), kode di sel tahap itu, dan versi pipeline. Jadi mengubah kode sebuah tahap juga membuat tahap itu dihitung ulang. Sidik disimpan di `keluaran/_meta/sNN.json` bersama daftar keluaran dan ringkasannya.

Saat **Run all**, sebuah tahap dilewati (diambil dari checkpoint) hanya bila:
1. semua berkas keluarannya masih ada,
2. sidiknya sama dengan sidik yang tersimpan, dan
3. kode tahap tidak tercantum di `paksa_ulang`.

Karena keluaran sebuah tahap menjadi masukan tahap berikutnya, perubahan di tahap awal otomatis membuat tahap-tahap sesudahnya dihitung ulang.

In [ ]:
STATUS_TAHAP = {}


def sha256_berkas(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for blok in iter(lambda: f.read(1 << 20), b""):
            h.update(blok)
    return h.hexdigest()


def sidik_objek(obj):
    teks = json.dumps(obj, sort_keys=True, ensure_ascii=False, default=str)
    return hashlib.sha256(teks.encode("utf-8")).hexdigest()


def sumber_sel_aktif():
    try:
        return get_ipython().user_ns["_ih"][-1]
    except Exception:
        return ""


def baca_csv(path):
    return pd.read_csv(path, keep_default_na=False, na_values=[""], dtype={"doc_id": str})


def simpan_csv(df, path):
    df.to_csv(path, index=False, encoding="utf-8")


class Tahap:
    def __init__(self, kode, nama, masukan, keluaran, parameter):
        self.kode = kode
        self.nama = nama
        self.masukan = [Path(p) for p in masukan]
        self.keluaran = [Path(p) for p in keluaran]
        self.parameter = parameter
        hilang = [str(p) for p in self.masukan if not p.exists()]
        if hilang:
            raise FileNotFoundError(f"[{kode}] Berkas masukan tidak ditemukan: {hilang}")
        self.sidik_masukan = {p.name: sha256_berkas(p) for p in self.masukan}
        self.sidik_kode = sidik_objek(sumber_sel_aktif())
        self.sidik = sidik_objek({
            "versi": KONFIGURASI["versi_pipeline"],
            "parameter": parameter,
            "masukan": self.sidik_masukan,
            "kode_sel": self.sidik_kode,
        })
        self.path_meta = DIR_META / f"{kode}.json"

    def bisa_dilewati(self):
        if self.kode in KONFIGURASI["paksa_ulang"]:
            return False
        if not self.path_meta.exists() or not all(p.exists() for p in self.keluaran):
            return False
        meta = json.loads(self.path_meta.read_text(encoding="utf-8"))
        if meta.get("sidik") != self.sidik:
            return False
        STATUS_TAHAP[self.kode] = "checkpoint"
        print(f"[{self.kode}] {self.nama}: diambil dari checkpoint ({meta['waktu']}).")
        for k, v in meta.get("ringkasan", {}).items():
            print(f"    {k}: {v}")
        return True

    def selesai(self, ringkasan):
        meta = {
            "kode": self.kode,
            "nama": self.nama,
            "sidik": self.sidik,
            "waktu": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "versi_pipeline": KONFIGURASI["versi_pipeline"],
            "parameter": self.parameter,
            "masukan": self.sidik_masukan,
            "keluaran": {p.name: sha256_berkas(p) for p in self.keluaran},
            "ringkasan": ringkasan,
        }
        self.path_meta.write_text(json.dumps(meta, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
        STATUS_TAHAP[self.kode] = "dijalankan"
        print(f"[{self.kode}] {self.nama}: selesai dan disimpan.")
        for k, v in ringkasan.items():
            print(f"    {k}: {v}")


def tampilkan(df, n=5):
    try:
        display(df.head(n))
    except NameError:
        print(df.head(n).to_string())

---
# Tahap 1. Muat dan validasi input anotator

**Tujuan:** memastikan berkas hasil kerja anotator terbaca utuh dan memberi setiap dokumen identitas yang tetap.

**Yang dikerjakan:**
1. Membaca `input/Combined Annotator.xlsx` (sheet pertama) apa adanya, tanpa mengubah isi teks.
2. Memeriksa bahwa kolom wajib tersedia: `Title`, `Text in Toba`, `Text in Bahasa Indonesia`, `Genre Category`, `Sub-Genre Category`.
3. Membuang baris yang benar-benar kosong (judul dan kedua teks kosong), lalu mencatat jumlahnya.
4. Memberi `doc_id` tetap dengan format `mg_001`, `mg_002`, dan seterusnya. Bila kolom `Column 1` berisi nomor urut yang unik, nomor itu dipakai. Bila tidak, dipakai urutan baris.
5. Memeriksa teks kosong, judul ganda, dan teks ganda, lalu mencatat sebaran genre dan sub-genre.
   Pemeriksaan juga mencari teks yang panjangnya tepat 32.767 karakter. Itu batas isi satu sel Excel, dan teks Agenda HKBP lebih panjang dari batas itu. Bila ada teks sepanjang itu, kemungkinan besar teksnya terpotong saat ekspor dari Google Sheet.
6. Menyimpan sidik SHA-256 berkas input agar versi data yang dipakai bisa dibuktikan.

**Masukan:** `input/Combined Annotator.xlsx`.

**Keluaran yang diharapkan:**
- `s01_dokumen_mentah.csv`: satu baris per dokumen (sekitar 104 dokumen), berisi semua kolom asli ditambah `doc_id`.
- `s01_validasi.json`: hasil pemeriksaan dan sidik berkas input.

**Dipakai untuk:** tahap 2. `doc_id` dari tahap ini dipakai sampai akhir pipeline, termasuk di `pair_id` korpus paralel.

**Cara memeriksa:** pastikan jumlah dokumen sesuai harapan, dan daftar `teks_bt_kosong` serta `teks_id_kosong` kosong. Bila tidak kosong, periksa baris tersebut di Google Sheet anotator.

**Perubahan dibanding v1:** di v1, `doc_id` bergantung pada urutan baris file v1.1 yang sudah digabung (Agenda HKBP 6 baris menjadi 1). Akibatnya `doc_0053` di korpus paralel tidak sama dengan baris 53 di `btb_multigenre.csv`. Di v2, `doc_id` mengikuti nomor dokumen anotator sehingga bisa ditelusuri langsung.

In [ ]:
P_S01_DOK = DIR_OUT / "s01_dokumen_mentah.csv"
P_S01_VAL = DIR_OUT / "s01_validasi.json"

if not BERKAS_ANOTATOR.exists():
    raise FileNotFoundError(
        f"Letakkan '{KONFIGURASI['berkas_anotator']}' di folder {DIR_INPUT} lalu jalankan ulang."
    )

t1 = Tahap(
    "s01", "Muat dan validasi input anotator",
    masukan=[BERKAS_ANOTATOR],
    keluaran=[P_S01_DOK, P_S01_VAL],
    parameter={k: KONFIGURASI[k] for k in ("sheet_anotator", "kolom_wajib")},
)

if not t1.bisa_dilewati():
    mentah = pd.read_excel(BERKAS_ANOTATOR, sheet_name=KONFIGURASI["sheet_anotator"], dtype=object)
    mentah.columns = [str(c).strip() for c in mentah.columns]

    hilang = [c for c in KONFIGURASI["kolom_wajib"] if c not in mentah.columns]
    if hilang:
        raise ValueError(f"Kolom wajib tidak ditemukan: {hilang}. Kolom yang ada: {list(mentah.columns)}")

    def kosong(nilai):
        return pd.isna(nilai) or not str(nilai).strip()

    kolom_isi = ["Title", "Text in Toba", "Text in Bahasa Indonesia"]
    baris_kosong = mentah[kolom_isi].apply(lambda r: all(kosong(v) for v in r), axis=1)
    mentah = mentah[~baris_kosong].reset_index(drop=True)

    sumber_id = "urutan_baris"
    if "Column 1" in mentah.columns:
        nomor = pd.to_numeric(mentah["Column 1"], errors="coerce")
        if nomor.notna().all() and (nomor == nomor.round()).all() and nomor.is_unique:
            mentah.insert(0, "doc_id", [f"mg_{int(n):03d}" for n in nomor])
            sumber_id = "Column 1"
    if sumber_id == "urutan_baris":
        mentah.insert(0, "doc_id", [f"mg_{i + 1:03d}" for i in range(len(mentah))])

    for kol in mentah.columns:
        mentah[kol] = mentah[kol].map(lambda v: v if pd.isna(v) else str(v))

    judul = mentah["Title"].fillna("").str.strip()
    batas_excel = 32767
    terpotong = [
        f"{r['doc_id']}:{kol}"
        for _, r in mentah.iterrows()
        for kol in ("Text in Toba", "Text in Bahasa Indonesia")
        if isinstance(r[kol], str) and len(r[kol]) == batas_excel
    ]
    validasi = {
        "berkas_input": BERKAS_ANOTATOR.name,
        "sha256_input": t1.sidik_masukan[BERKAS_ANOTATOR.name],
        "kolom_ditemukan": list(mentah.columns),
        "jumlah_dokumen": int(len(mentah)),
        "baris_kosong_dibuang": int(baris_kosong.sum()),
        "sumber_doc_id": sumber_id,
        "teks_bt_kosong": mentah.loc[mentah["Text in Toba"].map(kosong), "doc_id"].tolist(),
        "teks_id_kosong": mentah.loc[mentah["Text in Bahasa Indonesia"].map(kosong), "doc_id"].tolist(),
        "judul_ganda": sorted(judul[judul.duplicated(keep=False)].unique().tolist()),
        "teks_bt_ganda": mentah.loc[mentah["Text in Toba"].duplicated(keep=False), "doc_id"].tolist(),
        "dicurigai_terpotong_batas_excel": terpotong,
        "panjang_teks_maks": int(max(mentah["Text in Toba"].fillna("").str.len().max(),
                                     mentah["Text in Bahasa Indonesia"].fillna("").str.len().max())),
        "sebaran_genre": mentah["Genre Category"].value_counts().to_dict(),
        "sebaran_subgenre": mentah["Sub-Genre Category"].value_counts().to_dict(),
    }

    simpan_csv(mentah, P_S01_DOK)
    P_S01_VAL.write_text(json.dumps(validasi, indent=2, ensure_ascii=False), encoding="utf-8")
    t1.selesai({
        "jumlah_dokumen": validasi["jumlah_dokumen"],
        "baris_kosong_dibuang": validasi["baris_kosong_dibuang"],
        "sumber_doc_id": sumber_id,
        "teks_kosong": len(validasi["teks_bt_kosong"]) + len(validasi["teks_id_kosong"]),
        "dicurigai_terpotong": len(terpotong),
    })

dok_mentah = baca_csv(P_S01_DOK)
validasi_s01 = json.loads(P_S01_VAL.read_text(encoding="utf-8"))
print("\nSebaran genre:", validasi_s01["sebaran_genre"])
if validasi_s01["teks_bt_kosong"] or validasi_s01["teks_id_kosong"]:
    print("PERHATIAN: ada dokumen dengan teks kosong:", validasi_s01["teks_bt_kosong"], validasi_s01["teks_id_kosong"])
if validasi_s01["dicurigai_terpotong_batas_excel"]:
    print("PERHATIAN: teks berikut panjangnya tepat 32.767 karakter (batas sel Excel) dan kemungkinan terpotong:",
          validasi_s01["dicurigai_terpotong_batas_excel"])
tampilkan(dok_mentah[["doc_id", "Title", "Genre Category", "Sub-Genre Category"]])

---
# Tahap 2. Koreksi metadata tercatat

**Tujuan:** menyeragamkan struktur data dan memperbaiki kesalahan label secara terbuka, sehingga setiap perubahan bisa dipertanggungjawabkan.

**Yang dikerjakan:**
1. Mengganti nama kolom Google Sheet menjadi nama kolom baku (`title`, `text_bt`, `text_id`, `genre`, `subgenre`, `source_link`, `notes`, dan seterusnya) sesuai `kolom_anotator` di konfigurasi.
2. Menyeragamkan penulisan label, misalnya `Historical/ Traditional` menjadi `Historical/Traditional`.
3. Mengosongkan **isian contoh templat**. Enam baris pertama milik setiap anotator masih berisi metadata contoh dari templat Google Sheet (penulis "Dr. Sari Dewi", "Local Council", "Tempo Magazine"; catatan "Used for online course."; serta tanggal contoh). Nilai-nilai ini didaftar di `isian_templat` pada konfigurasi.
4. Mengosongkan `source_link` yang berisi penanda sementara seperti `[URL]`.
5. Memindahkan URL yang tertulis di kolom penulis ke kolom sumber (bila kolom sumber kosong), dan menghapus catatan yang hanya mengulang URL sumber.
6. Menerapkan daftar `koreksi_metadata`, termasuk 12 label genre atau sub-genre milik Annotator 2 yang tidak sesuai dengan daftar sumber di notebook panduan anotator (3 ulasan buku dan 9 abstrak artikel jurnal). Setiap judul dalam daftar harus cocok dengan **tepat satu** dokumen. Bila tidak, notebook berhenti dengan pesan galat agar tidak ada koreksi yang salah sasaran.
7. Memberi tanda `sumber_kosong` dan `notes_berisi_url`, lalu menampilkan semua isi `notes` yang tersisa agar peneliti bisa meninjaunya.

**Masukan:** `s01_dokumen_mentah.csv`.

**Keluaran yang diharapkan:**
- `s02_dokumen.csv`: tabel dokumen dengan kolom baku dan kolom `catatan_pipeline`.
- `s02_log_koreksi.csv`: satu baris per perubahan (doc_id, kolom, nilai lama, nilai baru, alasan).

**Dipakai untuk:** tahap 3. Log koreksi dipakai untuk bagian metodologi paper dan untuk menjawab pertanyaan reviewer tentang perubahan data.

**Cara memeriksa:** baca `s02_log_koreksi.csv`. Setiap baris harus punya alasan yang jelas. Periksa juga daftar isi `notes` yang ditampilkan di bawah sel.

**Perubahan dibanding v1:** di v1 hanya label Ikigai yang diperbaiki, itu pun hanya di v1.1 dan tidak sampai ke `btb_multigenre.csv`. Sebelas label lain yang salah, serta metadata contoh dari templat, ikut terbawa ke berkas rilis v1. Sekarang semua perbaikan itu menjadi langkah tercatat yang berlaku untuk semua keluaran.

In [ ]:
P_S02_DOK = DIR_OUT / "s02_dokumen.csv"
P_S02_LOG = DIR_OUT / "s02_log_koreksi.csv"

t2 = Tahap(
    "s02", "Koreksi metadata tercatat",
    masukan=[P_S01_DOK],
    keluaran=[P_S02_DOK, P_S02_LOG],
    parameter={k: KONFIGURASI[k] for k in ("kolom_anotator", "koreksi_metadata", "isian_templat", "normalisasi_label", "placeholder_sumber")},
)

if not t2.bisa_dilewati():
    dok = baca_csv(P_S01_DOK).rename(columns=KONFIGURASI["kolom_anotator"])
    kolom_baku = ["doc_id", "nomor_anotator", "date_collected", "title", "text_bt", "text_id", "genre",
                  "subgenre", "original_author", "source_link", "notes"]
    for kol in kolom_baku:
        if kol not in dok.columns:
            dok[kol] = np.nan
    dok = dok[kolom_baku].copy()
    dok["catatan_pipeline"] = ""
    log = []

    def catat(doc_id, kolom, lama, baru, alasan):
        log.append({"doc_id": doc_id, "kolom": kolom, "nilai_lama": lama, "nilai_baru": baru, "alasan": alasan})

    def tandai(idx, label):
        sekarang = dok.at[idx, "catatan_pipeline"]
        dok.at[idx, "catatan_pipeline"] = f"{sekarang};{label}".strip(";")

    for kol in ("title", "genre", "subgenre"):
        dok[kol] = dok[kol].fillna("").astype(str).str.strip()

    for kol, peta in KONFIGURASI["normalisasi_label"].items():
        for idx, nilai in dok[kol].items():
            if nilai in peta:
                dok.at[idx, kol] = peta[nilai]
                catat(dok.at[idx, "doc_id"], kol, nilai, peta[nilai], "Penyeragaman penulisan label")

    for kol, daftar in KONFIGURASI["isian_templat"].items():
        nilai_templat = {str(v).strip() for v in daftar}
        for idx, nilai in dok[kol].items():
            if isinstance(nilai, str) and nilai.strip() in nilai_templat:
                catat(dok.at[idx, "doc_id"], kol, nilai, "", "Isian contoh dari templat Google Sheet, bukan metadata asli")
                dok.at[idx, kol] = np.nan

    placeholder = {p.lower() for p in KONFIGURASI["placeholder_sumber"]}
    for idx, nilai in dok["source_link"].items():
        if isinstance(nilai, str) and nilai.strip().lower() in placeholder:
            catat(dok.at[idx, "doc_id"], "source_link", nilai, "", "Penanda sementara, bukan tautan sumber")
            dok.at[idx, "source_link"] = np.nan

    pola_url = re.compile(r"^\s*https?://\S+\s*$")
    for idx in dok.index:
        penulis = dok.at[idx, "original_author"]
        if isinstance(penulis, str) and pola_url.match(penulis):
            if pd.isna(dok.at[idx, "source_link"]):
                dok.at[idx, "source_link"] = penulis.strip()
                catat(dok.at[idx, "doc_id"], "source_link", "", penulis.strip(), "URL sumber tertulis di kolom penulis, dipindahkan ke kolom sumber")
            catat(dok.at[idx, "doc_id"], "original_author", penulis, "", "Kolom penulis berisi URL, bukan nama penulis")
            dok.at[idx, "original_author"] = np.nan
        catatan = dok.at[idx, "notes"]
        sumber = dok.at[idx, "source_link"]
        if isinstance(catatan, str) and isinstance(sumber, str) and catatan.strip() == sumber.strip():
            catat(dok.at[idx, "doc_id"], "notes", catatan, "", "Catatan hanya mengulang URL sumber")
            dok.at[idx, "notes"] = np.nan

    for k in KONFIGURASI["koreksi_metadata"]:
        judul_list = k["judul_mengandung"] if isinstance(k["judul_mengandung"], list) else [k["judul_mengandung"]]
        for judul in judul_list:
            cocok = dok.index[dok["title"].str.contains(judul, case=False, regex=False)]
            if len(cocok) != 1:
                raise ValueError(f"Koreksi '{judul}' cocok dengan {len(cocok)} dokumen, harus tepat 1.")
            idx = cocok[0]
            lama = dok.at[idx, k["kolom"]]
            if lama != k["nilai_baru"]:
                dok.at[idx, k["kolom"]] = k["nilai_baru"]
                catat(dok.at[idx, "doc_id"], k["kolom"], lama, k["nilai_baru"], k["alasan"])

    for idx in dok.index:
        if pd.isna(dok.at[idx, "source_link"]):
            tandai(idx, "sumber_kosong")
        if isinstance(dok.at[idx, "notes"], str) and re.search(r"https?://", dok.at[idx, "notes"]):
            tandai(idx, "notes_berisi_url")

    simpan_csv(dok, P_S02_DOK)
    log_df = pd.DataFrame(log, columns=["doc_id", "kolom", "nilai_lama", "nilai_baru", "alasan"])
    simpan_csv(log_df, P_S02_LOG)
    t2.selesai({
        "jumlah_perubahan": len(log_df),
        "dokumen_sumber_kosong": int(dok["catatan_pipeline"].str.contains("sumber_kosong").sum()),
        "dokumen_notes_berisi_url": int(dok["catatan_pipeline"].str.contains("notes_berisi_url").sum()),
    })

dok_s02 = baca_csv(P_S02_DOK)
log_s02 = baca_csv(P_S02_LOG)
tampilkan(log_s02, 20)
print("\nIsi kolom notes yang berbeda (perlu ditinjau peneliti):")
for nilai, jumlah in dok_s02["notes"].dropna().value_counts().items():
    print(f"  [{jumlah}x] {nilai}")

---
# Tahap 3. Pembersihan ringan dan berkas rilis multi-genre

**Tujuan:** merapikan aspek teknis teks tanpa mengubah kata, ejaan, kapitalisasi, atau tanda baca, lalu menghasilkan berkas rilis `btb_multigenre.csv`.

**Yang dikerjakan:**
1. Normalisasi Unicode ke bentuk NFC, sehingga huruf yang tampak sama tersimpan dengan kode yang sama.
2. Menyeragamkan akhir baris (`\r\n` menjadi `\n`), mengganti spasi tak terputus menjadi spasi biasa, dan menghapus karakter tak terlihat (zero-width).
3. Merapikan spasi berlebih dan baris kosong lebih dari dua, lalu memangkas spasi di awal dan akhir teks.
4. Mencatat dokumen mana yang berubah dan berapa karakter yang terdampak.
5. Menyusun berkas rilis dengan skema repositori: `doc_id`, `title`, `text_bt`, `text_id`, `genre`, `subgenre`, `source_link`, `notes`, `is_parallel`.

**Yang sengaja tidak dikerjakan:** huruf kecil, penghapusan tanda baca, dan perbaikan ejaan. Ketiganya mengubah data linguistik dan tidak boleh dilakukan pada berkas rilis. Versi huruf kecil tanpa tanda baca dibuat per kalimat di tahap 7 (kolom `teks_bt_norm` dan `teks_id_norm`).

**Masukan:** `s02_dokumen.csv`.

**Keluaran yang diharapkan:**
- `s03_btb_multigenre.csv`: berkas rilis korpus multi-genre, satu baris per dokumen.
- `s03_log_pembersihan.csv`: dokumen yang berubah beserta jumlah karakter sebelum dan sesudah.

**Dipakai untuk:** tahap 4 dan 5, serta disalin ke `rilis/data/btb_multigenre.csv` untuk repositori GitHub.

**Cara memeriksa:** selisih karakter per dokumen seharusnya kecil (hanya spasi dan karakter tak terlihat). Selisih besar menandakan ada masalah format di Google Sheet.

In [ ]:
P_S03_RILIS = DIR_OUT / "s03_btb_multigenre.csv"
P_S03_LOG = DIR_OUT / "s03_log_pembersihan.csv"

KARAKTER_TAK_TERLIHAT = re.compile(r"[​‌‍⁠﻿­]")


def bersihkan_teks(teks):
    if not isinstance(teks, str):
        return ""
    t = unicodedata.normalize("NFC", teks)
    t = t.replace("\r\n", "\n").replace("\r", "\n")
    t = t.replace(" ", " ").replace(" ", " ")
    t = KARAKTER_TAK_TERLIHAT.sub("", t)
    t = re.sub(r"[ \t]+\n", "\n", t)
    t = re.sub(r"\n[ \t]+", "\n", t)
    t = re.sub(r"[ \t]{2,}", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()


t3 = Tahap(
    "s03", "Pembersihan ringan",
    masukan=[P_S02_DOK],
    keluaran=[P_S03_RILIS, P_S03_LOG],
    parameter={"fungsi": "bersihkan_teks v1: NFC, akhir baris, spasi, karakter tak terlihat"},
)

if not t3.bisa_dilewati():
    dok = baca_csv(P_S02_DOK)
    log = []
    for kol in ("text_bt", "text_id"):
        sebelum = dok[kol].fillna("").astype(str)
        sesudah = sebelum.map(bersihkan_teks)
        for doc_id, a, b in zip(dok["doc_id"], sebelum, sesudah):
            if a != b:
                log.append({"doc_id": doc_id, "kolom": kol, "karakter_sebelum": len(a),
                            "karakter_sesudah": len(b), "selisih": len(a) - len(b),
                            "token_spasi_sama": len(a.split()) == len(b.split())})
        dok[kol] = sesudah

    dok["is_parallel"] = np.where((dok["text_bt"].str.len() > 0) & (dok["text_id"].str.len() > 0), "yes", "no")
    rilis = dok[["doc_id", "title", "text_bt", "text_id", "genre", "subgenre", "source_link", "notes", "is_parallel"]]
    simpan_csv(rilis, P_S03_RILIS)
    log_df = pd.DataFrame(log, columns=["doc_id", "kolom", "karakter_sebelum", "karakter_sesudah", "selisih", "token_spasi_sama"])
    simpan_csv(log_df, P_S03_LOG)
    t3.selesai({
        "dokumen": len(rilis),
        "sel_teks_berubah": len(log_df),
        "total_karakter_dirapikan": int(log_df["selisih"].sum()) if len(log_df) else 0,
        "jumlah_token_spasi_berubah": int((~log_df["token_spasi_sama"]).sum()) if len(log_df) else 0,
    })

dok_rilis = baca_csv(P_S03_RILIS)
log_s03 = baca_csv(P_S03_LOG)
tampilkan(log_s03.sort_values("selisih", ascending=False) if len(log_s03) else log_s03, 10)

---
# Tahap 4. Segmentasi kalimat

**Tujuan:** memecah setiap dokumen menjadi unit kalimat (atau baris, untuk puisi, umpasa, dan teks liturgi) di kedua bahasa, sebagai bahan penyelarasan.

**Yang dikerjakan:**
1. Melindungi angka desimal (misalnya `2.5`) agar titiknya tidak dianggap akhir kalimat.
2. Memecah teks pada tanda `.`, `!`, atau `?` yang diikuti spasi dan huruf kapital atau tanda kutip pembuka.
3. Menggabungkan kembali pecahan yang terjadi setelah singkatan (misalnya `Dr.`, `Prof.`, `Op.`). Daftar singkatan ada di konfigurasi.
4. Memecah lagi pada setiap baris baru, karena puisi, umpasa, dan agenda ditulis per baris.
5. Merapikan spasi di dalam setiap kalimat.

Fungsi pemecah ini sama dengan notebook segmentasi v1, sehingga perbedaan hasil v1 dan v2 tidak berasal dari tahap ini.

**Masukan:** `s03_btb_multigenre.csv`.

**Keluaran yang diharapkan:**
- `s04_kalimat.csv`: format panjang, satu baris per kalimat (`doc_id`, `bahasa` bernilai `bbc` atau `ind`, `idx`, `teks`).
- `s04_ringkasan_segmentasi.csv`: jumlah kalimat Toba dan Indonesia per dokumen beserta selisihnya.

**Dipakai untuk:** tahap 5 (jumlah kalimat) dan tahap 6 (penyelarasan).

**Cara memeriksa:** dokumen dengan selisih jumlah kalimat besar akan lebih sulit diselaraskan dan kemungkinan besar muncul di antrean review.

In [ ]:
P_S04_KAL = DIR_OUT / "s04_kalimat.csv"
P_S04_RING = DIR_OUT / "s04_ringkasan_segmentasi.csv"

PARAM_SEG = KONFIGURASI["segmentasi"]
SINGKATAN = set(PARAM_SEG["singkatan"])
POLA_AKHIR_KALIMAT = re.compile(r'(?<=[.!?])[\s\n]+(?=[A-ZÀ-Ý"\'“‘])')
POLA_DESIMAL = re.compile(r"\d+\.\d+")


def akhir_singkatan(teks_sebelum_titik):
    kata = re.findall(r"\b\w+\b", teks_sebelum_titik.lower())
    return bool(kata) and kata[-1] in SINGKATAN


def pecah_kalimat(teks, pisah_baris=True):
    if not isinstance(teks, str) or not teks.strip():
        return []
    simpanan = {}

    def lindungi(m):
        kunci = f"__DES{len(simpanan)}__"
        simpanan[kunci] = m.group(0)
        return kunci

    terlindung = POLA_DESIMAL.sub(lindungi, teks)
    kalimat, penampung = [], ""
    for potong in POLA_AKHIR_KALIMAT.split(terlindung):
        potong = potong.strip()
        if not potong:
            continue
        if penampung:
            if akhir_singkatan(penampung.rstrip(".!?").rstrip()):
                penampung = penampung + " " + potong
                continue
            kalimat.append(penampung)
        penampung = potong
    if penampung:
        kalimat.append(penampung)

    hasil = []
    for k in kalimat:
        bagian = re.split(r"\n+", k) if pisah_baris else re.split(r"\n{2,}", k)
        hasil.extend(b.strip() for b in bagian if b.strip())
    hasil = [re.sub(r"__DES\d+__", lambda m: simpanan.get(m.group(0), m.group(0)), k) for k in hasil]
    hasil = [re.sub(r"\s+", " ", k).strip() for k in hasil]
    return [k for k in hasil if k]


t4 = Tahap(
    "s04", "Segmentasi kalimat",
    masukan=[P_S03_RILIS],
    keluaran=[P_S04_KAL, P_S04_RING],
    parameter=PARAM_SEG,
)

if not t4.bisa_dilewati():
    dok = baca_csv(P_S03_RILIS)
    baris, ringkasan = [], []
    for _, r in dok.iterrows():
        jumlah = {}
        for bahasa, kol in (("bbc", "text_bt"), ("ind", "text_id")):
            daftar = pecah_kalimat(r[kol], PARAM_SEG["pisah_setiap_baris_baru"])
            jumlah[bahasa] = len(daftar)
            baris.extend({"doc_id": r["doc_id"], "bahasa": bahasa, "idx": i, "teks": k} for i, k in enumerate(daftar))
        ringkasan.append({"doc_id": r["doc_id"], "title": r["title"], "genre": r["genre"], "subgenre": r["subgenre"],
                          "kalimat_bt": jumlah["bbc"], "kalimat_id": jumlah["ind"],
                          "selisih": abs(jumlah["bbc"] - jumlah["ind"])})
    kal = pd.DataFrame(baris)
    ring = pd.DataFrame(ringkasan)
    simpan_csv(kal, P_S04_KAL)
    simpan_csv(ring, P_S04_RING)
    t4.selesai({
        "kalimat_bt": int(ring["kalimat_bt"].sum()),
        "kalimat_id": int(ring["kalimat_id"].sum()),
        "dokumen_jumlah_kalimat_sama": int((ring["selisih"] == 0).sum()),
    })

kalimat = baca_csv(P_S04_KAL)
ring_seg = baca_csv(P_S04_RING)
tampilkan(ring_seg.sort_values("selisih", ascending=False), 10)

---
# Tahap 5. Statistik korpus

**Tujuan:** menghasilkan angka resmi korpus dengan metode hitung yang **dinyatakan secara eksplisit**, sehingga angka di README, laporan teknis, dan paper konsisten dan bisa diperiksa ulang.

**Yang dikerjakan:**
1. Menghitung token dengan tiga metode untuk setiap dokumen dan bahasa:
   - `token_spasi`: pemisahan berdasarkan spasi pada teks rilis. Tanda baca yang menempel pada kata tidak dihitung terpisah.
   - `kata`: hanya urutan huruf atau angka (`\w+`). Tanda baca tidak dihitung. Ini angka yang paling tepat disebut "jumlah kata".
   - `token_nltk_v1`: metode EDA versi lama, yaitu teks dijadikan huruf kecil, dibersihkan, lalu dipecah dengan NLTK `word_tokenize`. **Tanda baca ikut dihitung sebagai token.** Metode ini disertakan hanya agar angka lama (112.340 token) bisa direproduksi dan dibandingkan.
2. Menghitung jumlah kalimat dari hasil tahap 4.
3. Merangkum per genre dan sub-genre, termasuk porsi teks religius dan porsi Agenda HKBP.
4. Memeriksa target 100.000 token untuk setiap metode secara terpisah.
5. Menyusun potongan tabel Markdown siap tempel untuk README.

**Masukan:** `s03_btb_multigenre.csv` dan `s04_ringkasan_segmentasi.csv`.

**Keluaran yang diharapkan:**
- `s05_statistik.json`: total per metode, status target, porsi genre.
- `s05_statistik_per_genre.csv` dan `s05_statistik_per_dokumen.csv`.
- `s05_tabel_readme.md`: tabel siap pakai untuk README dan laporan teknis.

**Dipakai untuk:** README repositori, laporan teknis, bagian statistik korpus di paper.

**Cara memeriksa:** bandingkan `token_nltk_v1` dengan angka lama 67.680 (sisi Toba multi-genre). Bila sama, metode lama berhasil direproduksi. Saat melaporkan, sebutkan metode yang dipakai.

In [ ]:
P_S05_JSON = DIR_OUT / "s05_statistik.json"
P_S05_GENRE = DIR_OUT / "s05_statistik_per_genre.csv"
P_S05_DOK = DIR_OUT / "s05_statistik_per_dokumen.csv"
P_S05_MD = DIR_OUT / "s05_tabel_readme.md"

import nltk
for paket in ("punkt", "punkt_tab"):
    try:
        nltk.download(paket, quiet=True)
    except Exception as e:
        print("Gagal mengunduh", paket, e)
from nltk.tokenize import sent_tokenize, word_tokenize


def bersihkan_gaya_eda_v1(teks):
    t = teks.lower()
    t = re.sub(r"[^a-z0-9.,!?;:\s]", " ", t)
    t = re.sub(r"\s+", " ", t)
    return t.strip()


def token_nltk_v1(teks):
    t = bersihkan_gaya_eda_v1(teks)
    return sum(len(word_tokenize(k)) for p in t.split("\n") for k in sent_tokenize(p)) if t else 0


t5 = Tahap(
    "s05", "Statistik korpus",
    masukan=[P_S03_RILIS, P_S04_RING],
    keluaran=[P_S05_JSON, P_S05_GENRE, P_S05_DOK, P_S05_MD],
    parameter={"target_token": KONFIGURASI["target_token"], "metode": ["token_spasi", "kata", "token_nltk_v1"],
               "subgenre_liturgi": KONFIGURASI["penyelarasan"]["subgenre_liturgi"]},
)

if not t5.bisa_dilewati():
    dok = baca_csv(P_S03_RILIS)
    seg = baca_csv(P_S04_RING)[["doc_id", "kalimat_bt", "kalimat_id"]]
    rows = []
    for _, r in dok.iterrows():
        baris = {"doc_id": r["doc_id"], "title": r["title"], "genre": r["genre"], "subgenre": r["subgenre"]}
        for suf, kol in (("bt", "text_bt"), ("id", "text_id")):
            teks = r[kol] if isinstance(r[kol], str) else ""
            baris[f"token_spasi_{suf}"] = len(teks.split())
            baris[f"kata_{suf}"] = len(re.findall(r"\w+", teks))
            baris[f"token_nltk_v1_{suf}"] = token_nltk_v1(teks)
        rows.append(baris)
    per_dok = pd.DataFrame(rows).merge(seg, on="doc_id", how="left")
    metrik = [c for c in per_dok.columns if c.startswith(("token_", "kata_", "kalimat_"))]

    per_genre = per_dok.groupby("genre")[metrik].sum()
    per_genre.insert(0, "dokumen", per_dok.groupby("genre").size())
    per_genre["porsi_kata_bt_persen"] = (100 * per_genre["kata_bt"] / per_genre["kata_bt"].sum()).round(2)
    per_genre = per_genre.reset_index()

    total = {c: int(per_dok[c].sum()) for c in metrik}
    target = KONFIGURASI["target_token"]
    liturgi = per_dok["subgenre"].isin(KONFIGURASI["penyelarasan"]["subgenre_liturgi"])
    religius = per_dok["genre"] == "Religious"
    statistik = {
        "jumlah_dokumen": int(len(per_dok)),
        "total": total,
        "target_token": target,
        "target_tercapai_sisi_bt": {
            "token_spasi": total["token_spasi_bt"] >= target,
            "kata": total["kata_bt"] >= target,
            "token_nltk_v1": total["token_nltk_v1_bt"] >= target,
        },
        "porsi_religius_kata_bt_persen": round(100 * per_dok.loc[religius, "kata_bt"].sum() / total["kata_bt"], 2),
        "porsi_liturgi_kata_bt_persen": round(100 * per_dok.loc[liturgi, "kata_bt"].sum() / total["kata_bt"], 2),
        "catatan_metode": {
            "token_spasi": "split() pada teks rilis; tanda baca menempel pada kata",
            "kata": "regex \\w+; tanda baca tidak dihitung",
            "token_nltk_v1": "huruf kecil, pembersihan gaya EDA v1, NLTK word_tokenize; tanda baca dihitung sebagai token",
        },
    }

    def fmt(n):
        return f"{n:,}".replace(",", ".")

    tabel = [
        "| Ukuran (sub-korpus multi-genre) | Batak Toba | Bahasa Indonesia |",
        "|---|---|---|",
        f"| Dokumen | {fmt(len(per_dok))} | {fmt(len(per_dok))} |",
        f"| Kalimat atau baris (segmentasi tahap 4) | {fmt(total['kalimat_bt'])} | {fmt(total['kalimat_id'])} |",
        f"| Kata (tanpa tanda baca) | {fmt(total['kata_bt'])} | {fmt(total['kata_id'])} |",
        f"| Token berbasis spasi | {fmt(total['token_spasi_bt'])} | {fmt(total['token_spasi_id'])} |",
        f"| Token NLTK, termasuk tanda baca (metode v1) | {fmt(total['token_nltk_v1_bt'])} | {fmt(total['token_nltk_v1_id'])} |",
    ]
    simpan_csv(per_dok, P_S05_DOK)
    simpan_csv(per_genre, P_S05_GENRE)
    P_S05_JSON.write_text(json.dumps(statistik, indent=2, ensure_ascii=False), encoding="utf-8")
    P_S05_MD.write_text("\n".join(tabel) + "\n", encoding="utf-8")
    t5.selesai({
        "kata_bt": total["kata_bt"],
        "token_spasi_bt": total["token_spasi_bt"],
        "token_nltk_v1_bt": total["token_nltk_v1_bt"],
        "porsi_religius_persen": statistik["porsi_religius_kata_bt_persen"],
    })

statistik_s05 = json.loads(P_S05_JSON.read_text(encoding="utf-8"))
print(P_S05_MD.read_text(encoding="utf-8"))
print("Target tercapai (sisi Batak Toba):", statistik_s05["target_tercapai_sisi_bt"])
print("Porsi religius:", statistik_s05["porsi_religius_kata_bt_persen"], "% | porsi liturgi (Agenda HKBP):",
      statistik_s05["porsi_liturgi_kata_bt_persen"], "%")
tampilkan(baca_csv(P_S05_GENRE), 10)

---
# Tahap 6. Penyelarasan kalimat

**Tujuan:** memasangkan kalimat Batak Toba dengan padanan Indonesianya di setiap dokumen.

**Yang dikerjakan:**
1. Menentukan korpus tujuan: dokumen bersub-genre `Ritual Text` (Agenda HKBP) masuk **korpus liturgi**, dokumen lain masuk **korpus utama**. Keenam bagian Agenda HKBP diselaraskan **per bagian**, karena setiap bagian Toba sudah berpasangan dengan bagian Indonesianya.
2. Memilih strategi per dokumen:
   - `rule` (pasangan 1:1 berdasarkan urutan) untuk sub-genre yang strukturnya sejajar baris per baris (`Prayers`, `Peribahasa/Umpama`), **hanya bila jumlah kalimat kedua sisi sama**.
   - Bila jumlah kalimat tidak sama, dokumen tersebut dialihkan ke LaBSE dan dicatat sebagai `labse_alih`.
   - `labse` untuk semua sub-genre lain.
3. Untuk strategi LaBSE: setiap kalimat diubah menjadi vektor oleh model LaBSE, lalu algoritme pemrograman dinamis (mengikuti pendekatan Vecalign/Bertalign) mencari susunan blok terbaik dengan tipe 1-1, 1-2, 2-1, 2-2, 1-3, 3-1, serta 1-0 dan 0-1 untuk kalimat tanpa pasangan.
4. Menyimpan **semua** blok, termasuk blok tanpa pasangan (1-0 dan 0-1), agar cakupan penyelarasan bisa diukur.

**Catatan metodologis penting:**
- LaBSE tidak dilatih secara khusus dengan bahasa Batak Toba. Skor kemiripan (`sim_labse`) untuk pasangan Toba dan Indonesia cenderung rendah meskipun pasangannya benar. Karena itu skor ini dipakai sebagai salah satu sinyal di tahap audit, bukan sebagai bukti benar atau salahnya terjemahan.
- Model berjalan jauh lebih cepat dengan GPU. Pada CPU, tahap ini bisa memakan waktu puluhan menit.

**Masukan:** `s03_btb_multigenre.csv` dan `s04_kalimat.csv`.

**Keluaran yang diharapkan:**
- `s06_penyelarasan.csv`: satu baris per blok (`pair_id`, `doc_id`, `korpus`, `idx_bt`, `idx_id`, `teks_bt`, `teks_id`, `tipe`, `metode`, `sim_labse`, dan lainnya).
- `s06_ringkasan_penyelarasan.csv`: per dokumen, metode yang dipakai, sebaran tipe blok, dan cakupan kata Toba yang berhasil dipasangkan.

**Dipakai untuk:** tahap 7.

**Cara memeriksa:** lihat dokumen dengan `cakupan_kata_bt` rendah di ringkasan. Cakupan rendah berarti banyak kalimat Toba tidak menemukan pasangan.

**Perubahan dibanding v1:**
1. Di v1, strategi `rule` memotong sisa kalimat secara diam-diam bila jumlah kalimat tidak sama. Di v2, dokumen seperti itu dialihkan ke LaBSE.
2. Di v1, blok tanpa pasangan dibuang tanpa jejak. Di v2, blok itu disimpan dan dilaporkan.
3. Di v1, Agenda HKBP digabung menjadi satu dokumen besar lalu dikeluarkan. Di v2, Agenda diselaraskan per bagian sebagai korpus liturgi.

In [ ]:
P_S06_PASANGAN = DIR_OUT / "s06_penyelarasan.csv"
P_S06_RING = DIR_OUT / "s06_ringkasan_penyelarasan.csv"
PARAM_AL = KONFIGURASI["penyelarasan"]


class PenyematLaBSE:
    def __init__(self, nama_model, ukuran_batch):
        import torch
        from sentence_transformers import SentenceTransformer
        perangkat = "cuda" if torch.cuda.is_available() else "cpu"
        if perangkat == "cpu":
            print("PERINGATAN: GPU tidak tersedia. Tahap ini akan lambat. Pilih Runtime > Change runtime type > T4 GPU.")
        torch.manual_seed(KONFIGURASI["seed"])
        self.model = SentenceTransformer(nama_model, device=perangkat)
        self.ukuran_batch = ukuran_batch
        self.perangkat = perangkat
        print(f"Model {nama_model} dimuat di {perangkat}.")

    def sematkan(self, kalimat):
        if not kalimat:
            return np.zeros((0, 768), dtype=np.float32)
        return self.model.encode(kalimat, batch_size=self.ukuran_batch, convert_to_numpy=True,
                                 normalize_embeddings=True, show_progress_bar=False).astype(np.float32)


TIPE_BLOK = [tuple(t) for t in PARAM_AL["tipe_blok"]]


def skor_blok(sim, i, j, di, dj):
    if di == 0 or dj == 0:
        return -PARAM_AL["penalti_null"]
    if di == 1 and dj == 1:
        return float(sim[i, j])
    return float(sim[i:i + di, j:j + dj].mean()) - PARAM_AL["penalti_gabung"] * ((di - 1) + (dj - 1))


def selaraskan_dp(sim):
    n, m = sim.shape
    NEG = -1e9
    dp = np.full((n + 1, m + 1), NEG)
    jejak = {}
    dp[0, 0] = 0.0
    for i in range(n + 1):
        for j in range(m + 1):
            if dp[i, j] == NEG:
                continue
            for di, dj in TIPE_BLOK:
                ni, nj = i + di, j + dj
                if ni > n or nj > m:
                    continue
                kandidat = dp[i, j] + skor_blok(sim, i, j, di, dj)
                if kandidat > dp[ni, nj]:
                    dp[ni, nj] = kandidat
                    jejak[(ni, nj)] = (i, j, di, dj)
    blok, i, j = [], n, m
    while (i, j) != (0, 0):
        pi, pj, di, dj = jejak[(i, j)]
        blok.append((list(range(pi, pi + di)), list(range(pj, pj + dj))))
        i, j = pi, pj
    return blok[::-1]


t6 = Tahap(
    "s06", "Penyelarasan kalimat",
    masukan=[P_S03_RILIS, P_S04_KAL],
    keluaran=[P_S06_PASANGAN, P_S06_RING],
    parameter=PARAM_AL,
)

if not t6.bisa_dilewati():
    dok = baca_csv(P_S03_RILIS)
    kal = baca_csv(P_S04_KAL).sort_values(["doc_id", "bahasa", "idx"])
    kelompok = {k: g["teks"].tolist() for k, g in kal.groupby(["doc_id", "bahasa"])}
    penyemat = PenyematLaBSE(PARAM_AL["model"], PARAM_AL["ukuran_batch"])

    baris, ringkasan = [], []
    for nomor, (_, r) in enumerate(dok.iterrows(), start=1):
        doc_id, sub = r["doc_id"], r["subgenre"]
        korpus = "liturgi" if sub in PARAM_AL["subgenre_liturgi"] else "utama"
        bt = kelompok.get((doc_id, "bbc"), [])
        idn = kelompok.get((doc_id, "ind"), [])
        strategi = PARAM_AL["strategi_per_subgenre"].get(sub, PARAM_AL["strategi_bawaan"])
        if not bt or not idn:
            ringkasan.append({"doc_id": doc_id, "korpus": korpus, "subgenre": sub, "metode": "kosong",
                              "kalimat_bt": len(bt), "kalimat_id": len(idn)})
            continue
        sim = penyemat.sematkan(bt) @ penyemat.sematkan(idn).T
        if strategi == "rule" and len(bt) == len(idn):
            metode, blok = "rule", [([k], [k]) for k in range(len(bt))]
        else:
            metode = "labse_alih" if strategi == "rule" else "labse"
            blok = selaraskan_dp(sim)

        hitung_tipe = {}
        kata_bt_total = sum(len(re.findall(r"\w+", s)) for s in bt)
        kata_bt_pasang = 0
        for urutan, (ib, ii) in enumerate(blok):
            tipe = f"{len(ib)}-{len(ii)}"
            hitung_tipe[tipe] = hitung_tipe.get(tipe, 0) + 1
            teks_bt = " ".join(bt[k] for k in ib)
            teks_id = " ".join(idn[k] for k in ii)
            berpasangan = bool(ib) and bool(ii)
            if berpasangan:
                kata_bt_pasang += len(re.findall(r"\w+", teks_bt))
            baris.append({
                "pair_id": f"{doc_id}_p{urutan:04d}",
                "doc_id": doc_id,
                "korpus": korpus,
                "urutan": urutan,
                "idx_bt": ",".join(map(str, ib)),
                "idx_id": ",".join(map(str, ii)),
                "teks_bt": teks_bt,
                "teks_id": teks_id,
                "tipe": tipe,
                "berpasangan": berpasangan,
                "metode": metode,
                "sim_labse": round(float(sim[np.ix_(ib, ii)].mean()), 4) if berpasangan else np.nan,
                "genre": r["genre"],
                "subgenre": sub,
            })
        ringkasan.append({"doc_id": doc_id, "korpus": korpus, "subgenre": sub, "metode": metode,
                          "kalimat_bt": len(bt), "kalimat_id": len(idn), "blok": len(blok),
                          "sebaran_tipe": json.dumps(hitung_tipe, sort_keys=True),
                          "cakupan_kata_bt": round(kata_bt_pasang / kata_bt_total, 4) if kata_bt_total else np.nan})
        if nomor % 10 == 0 or nomor == len(dok):
            print(f"  {nomor}/{len(dok)} dokumen selesai")

    pasangan = pd.DataFrame(baris)
    ring = pd.DataFrame(ringkasan)
    simpan_csv(pasangan, P_S06_PASANGAN)
    simpan_csv(ring, P_S06_RING)
    t6.selesai({
        "perangkat": penyemat.perangkat,
        "blok_total": len(pasangan),
        "blok_berpasangan": int(pasangan["berpasangan"].sum()),
        "blok_tanpa_pasangan": int((~pasangan["berpasangan"]).sum()),
        "metode": pasangan.groupby("metode")["doc_id"].nunique().to_dict(),
    })

pasangan_s06 = baca_csv(P_S06_PASANGAN)
ring_s06 = baca_csv(P_S06_RING)
print(pasangan_s06.groupby(["korpus", "tipe"]).size().unstack(fill_value=0))
tampilkan(ring_s06.sort_values("cakupan_kata_bt"), 10)

---
# Tahap 7. Audit otomatis dan pelabelan status

**Tujuan:** memberi setiap pasangan tanda masalah yang terukur, prioritas review, dan status yang jujur, sehingga pengguna korpus tahu mana pasangan yang aman dipakai dan mana yang perlu diperiksa manusia.

**Yang dikerjakan:**
1. Memisahkan blok tanpa pasangan (1-0 dan 0-1) ke berkas tersendiri.
2. Menghitung ukuran setiap pasangan: jumlah token, rasio panjang karakter, dan rasio jumlah token.
3. Memberi tanda (`flag_audit`) berdasarkan aturan berikut. Semua ambang ada di konfigurasi bagian `audit`.

| Tanda | Arti | Prioritas |
|---|---|---|
| `tipe_bukan_1_1` | Blok menggabungkan lebih dari satu kalimat di salah satu sisi | tinggi |
| `sisi_identik` | Teks Toba dan Indonesia sama persis setelah dinormalisasi dan lebih dari 3 token, kemungkinan bukan terjemahan | tinggi |
| `identik_pendek` | Sama persis tetapi maksimal 3 token, biasanya nama diri, judul, atau nama marga yang memang tidak diterjemahkan | sedang |
| `tanpa_huruf` | Salah satu sisi hanya berisi angka atau tanda baca, misalnya nomor butir "1." | sedang |
| `rasio_karakter_ekstrem` | Rasio panjang karakter Toba terhadap Indonesia di luar rentang 0,4 sampai 2,5 | tinggi |
| `rasio_token_ekstrem` | Jumlah token satu sisi minimal dua kali sisi lainnya | sedang |
| `unit_panjang` | Salah satu sisi lebih dari 45 token, batas kalimat mungkin terlewat | sedang |
| `sim_rendah` | Skor LaBSE di bawah 0,2 | sedang |
| `duplikat_persis` | Pasangan yang sama persis muncul lebih dari sekali | sedang |
| `tanya_tidak_cocok` | Hanya satu sisi yang diakhiri tanda tanya | sedang |
| `bersebelahan_tanpa_pasangan` | Bertetangga dengan blok tanpa pasangan, sehingga batasnya mungkin bergeser | sedang |
| `fragmen` | Unit sangat pendek tanpa tanda baca akhir, atau seluruhnya huruf kapital (judul) | sedang |

4. Menentukan `prioritas_review` (`tinggi`, `sedang`, `rutin`) dan `status_penyelarasan`: `perlu_review` untuk prioritas tinggi, `kandidat` untuk sisanya.
5. Mengisi `validasi_terjemahan = menunggu_manusia` untuk **semua** pasangan. Belum ada pasangan yang dinyatakan gold standard.
6. Membuat kolom versi normal (`teks_bt_norm`, `teks_id_norm`: huruf kecil, tanpa tanda baca umum) untuk keperluan pemodelan.

**Masukan:** `s06_penyelarasan.csv`.

**Keluaran yang diharapkan:**
- `s07_paralel_utama.csv`: korpus paralel tingkat kalimat, korpus utama.
- `s07_paralel_liturgi.csv`: korpus paralel tingkat kalimat, Agenda HKBP.
- `s07_tak_berpasangan.csv`: kalimat yang tidak menemukan pasangan.
- `s07_ringkasan_audit.json`: jumlah per status, prioritas, dan tanda.

**Dipakai untuk:** tahap 8 (pemilihan sampel review) dan rilis di `data/experimental/`.

**Cara memeriksa:** lihat contoh pasangan berprioritas tinggi di bawah sel. Tanda otomatis bukan bukti kesalahan terjemahan, hanya penunjuk tempat yang perlu dibaca manusia.

**Perubahan dibanding v1:** aturan audit v1 dijalankan setengah manual di luar notebook (1.494 baris masukannya tidak tercatat asal-usulnya). Di v2, aturannya tertulis di sini dan berlaku untuk seluruh pasangan.

In [ ]:
P_S07_UTAMA = DIR_OUT / "s07_paralel_utama.csv"
P_S07_LITURGI = DIR_OUT / "s07_paralel_liturgi.csv"
P_S07_TAK = DIR_OUT / "s07_tak_berpasangan.csv"
P_S07_JSON = DIR_OUT / "s07_ringkasan_audit.json"
PARAM_AU = KONFIGURASI["audit"]
POLA_TANDA_BACA_NORM = re.compile(r'[.,!?;:"“”‘’()\[\]]')


def normalkan_kalimat(teks):
    if not isinstance(teks, str):
        return ""
    return re.sub(r"\s+", " ", POLA_TANDA_BACA_NORM.sub(" ", teks.lower())).strip()


def tampak_fragmen(teks):
    huruf = re.sub(r"[^A-Za-zÀ-ÿ]", "", teks)
    semua_kapital = len(huruf) >= 4 and huruf.isupper()
    pendek = len(teks.split()) <= PARAM_AU["fragmen_maks_token"] and not re.search(r'[.!?]["”’)]*$', teks)
    return semua_kapital or pendek


t7 = Tahap(
    "s07", "Audit otomatis dan pelabelan status",
    masukan=[P_S06_PASANGAN],
    keluaran=[P_S07_UTAMA, P_S07_LITURGI, P_S07_TAK, P_S07_JSON],
    parameter=PARAM_AU,
)

if not t7.bisa_dilewati():
    semua = baca_csv(P_S06_PASANGAN).sort_values(["doc_id", "urutan"]).reset_index(drop=True)
    semua["berpasangan"] = semua["berpasangan"].astype(str).str.lower().eq("true")
    for kol in ("teks_bt", "teks_id"):
        semua[kol] = semua[kol].fillna("")

    tetangga_null = set()
    for _, g in semua.groupby("doc_id"):
        urut = g.sort_values("urutan")
        null_pos = urut.index[~urut["berpasangan"]]
        posisi = list(urut.index)
        for p in null_pos:
            k = posisi.index(p)
            tetangga_null.update(posisi[max(k - 1, 0):k] + posisi[k + 1:k + 2])

    tak = semua[~semua["berpasangan"]].copy()
    par = semua[semua["berpasangan"]].copy()
    par["teks_bt_norm"] = par["teks_bt"].map(normalkan_kalimat)
    par["teks_id_norm"] = par["teks_id"].map(normalkan_kalimat)
    par["token_bt"] = par["teks_bt"].str.split().str.len()
    par["token_id"] = par["teks_id"].str.split().str.len()
    par["rasio_karakter"] = (par["teks_bt"].str.len() / par["teks_id"].str.len().clip(lower=1)).round(3)
    par["rasio_token"] = (par["token_bt"] / par["token_id"].clip(lower=1)).round(3)
    duplikat = par.duplicated(subset=["teks_bt_norm", "teks_id_norm"], keep=False)

    def tanda_untuk(idx, r):
        t = []
        if r["tipe"] != "1-1":
            t.append("tipe_bukan_1_1")
        if r["teks_bt_norm"] and r["teks_bt_norm"] == r["teks_id_norm"]:
            pendek = max(r["token_bt"], r["token_id"]) <= PARAM_AU["fragmen_maks_token"]
            t.append("identik_pendek" if pendek else "sisi_identik")
        if not re.search(r"[^\W\d_]", r["teks_bt"]) or not re.search(r"[^\W\d_]", r["teks_id"]):
            t.append("tanpa_huruf")
        if not PARAM_AU["rasio_karakter_min"] <= r["rasio_karakter"] <= PARAM_AU["rasio_karakter_maks"]:
            t.append("rasio_karakter_ekstrem")
        rt = PARAM_AU["rasio_token_ekstrem"]
        if r["rasio_token"] >= rt or r["rasio_token"] <= 1 / rt:
            t.append("rasio_token_ekstrem")
        if max(r["token_bt"], r["token_id"]) > PARAM_AU["panjang_unit_maks"]:
            t.append("unit_panjang")
        if pd.notna(r["sim_labse"]) and float(r["sim_labse"]) < PARAM_AU["sim_rendah"]:
            t.append("sim_rendah")
        if duplikat.loc[idx]:
            t.append("duplikat_persis")
        if r["teks_bt"].rstrip().endswith("?") != r["teks_id"].rstrip().endswith("?"):
            t.append("tanya_tidak_cocok")
        if idx in tetangga_null:
            t.append("bersebelahan_tanpa_pasangan")
        if tampak_fragmen(r["teks_bt"]) or tampak_fragmen(r["teks_id"]):
            t.append("fragmen")
        return t

    tanda = [tanda_untuk(idx, r) for idx, r in par.iterrows()]
    tinggi = set(PARAM_AU["tanda_prioritas_tinggi"])
    par["flag_audit"] = [";".join(t) for t in tanda]
    par["prioritas_review"] = ["tinggi" if tinggi & set(t) else ("sedang" if t else "rutin") for t in tanda]
    par["jenis_unit"] = ["fragmen_atau_judul" if "fragmen" in t else "kandidat_kalimat" for t in tanda]
    par["status_penyelarasan"] = np.where(par["prioritas_review"] == "tinggi", "perlu_review", "kandidat")
    par["validasi_terjemahan"] = "menunggu_manusia"

    kolom = ["pair_id", "doc_id", "korpus", "urutan", "idx_bt", "idx_id", "teks_bt", "teks_id",
             "teks_bt_norm", "teks_id_norm", "genre", "subgenre", "metode", "tipe", "sim_labse",
             "token_bt", "token_id", "rasio_karakter", "rasio_token", "jenis_unit", "flag_audit",
             "prioritas_review", "status_penyelarasan", "validasi_terjemahan"]
    utama = par.loc[par["korpus"] == "utama", kolom]
    liturgi = par.loc[par["korpus"] == "liturgi", kolom]
    tak = tak[["pair_id", "doc_id", "korpus", "urutan", "idx_bt", "idx_id", "teks_bt", "teks_id", "tipe", "metode", "genre", "subgenre"]]

    def ringkas(df):
        hitung_tanda = {}
        for f in df["flag_audit"]:
            for x in filter(None, f.split(";")):
                hitung_tanda[x] = hitung_tanda.get(x, 0) + 1
        return {
            "pasangan": int(len(df)),
            "dokumen": int(df["doc_id"].nunique()),
            "kata_bt": int(df["teks_bt"].map(lambda s: len(re.findall(r"\w+", s))).sum()),
            "kata_id": int(df["teks_id"].map(lambda s: len(re.findall(r"\w+", s))).sum()),
            "status": df["status_penyelarasan"].value_counts().to_dict(),
            "prioritas": df["prioritas_review"].value_counts().to_dict(),
            "tipe": df["tipe"].value_counts().to_dict(),
            "metode": df["metode"].value_counts().to_dict(),
            "tanda": dict(sorted(hitung_tanda.items(), key=lambda kv: -kv[1])),
        }

    audit = {"utama": ringkas(utama), "liturgi": ringkas(liturgi),
             "tak_berpasangan": {"blok": int(len(tak)), "per_korpus": tak["korpus"].value_counts().to_dict()}}
    simpan_csv(utama, P_S07_UTAMA)
    simpan_csv(liturgi, P_S07_LITURGI)
    simpan_csv(tak, P_S07_TAK)
    P_S07_JSON.write_text(json.dumps(audit, indent=2, ensure_ascii=False), encoding="utf-8")
    t7.selesai({
        "pasangan_utama": audit["utama"]["pasangan"],
        "pasangan_liturgi": audit["liturgi"]["pasangan"],
        "perlu_review_utama": audit["utama"]["status"].get("perlu_review", 0),
        "blok_tak_berpasangan": audit["tak_berpasangan"]["blok"],
    })

paralel_utama = baca_csv(P_S07_UTAMA)
paralel_liturgi = baca_csv(P_S07_LITURGI)
audit_s07 = json.loads(P_S07_JSON.read_text(encoding="utf-8"))
print(json.dumps(audit_s07["utama"], indent=2, ensure_ascii=False))
print("\nContoh pasangan berprioritas tinggi (korpus utama):")
tampilkan(paralel_utama.loc[paralel_utama["prioritas_review"] == "tinggi",
                            ["pair_id", "tipe", "flag_audit", "teks_bt", "teks_id"]], 8)

---
# Tahap 8. Lembar review untuk dua anotator (siap IAA)

**Tujuan:** menyiapkan bahan pemeriksaan manusia yang bisa dipakai sekaligus untuk mengukur kesepakatan antaranotator.

**Yang dikerjakan:**
1. Menyusun sampel review dari korpus utama (korpus liturgi ikut bila `sertakan_liturgi = True`):
   - semua pasangan berprioritas **tinggi** dan **sedang**,
   - ditambah sampel acak pasangan **rutin** sebanyak 10% (minimal 100 pasangan) dengan seed tetap.

   Sampel rutin penting: tanpa itu, IAA hanya mengukur kasus sulit dan kualitas pasangan yang lolos audit otomatis tidak pernah diuji.
2. Mengacak urutan sampel dengan seed tetap, agar anotator tidak membaca kasus sulit secara berkelompok.
3. **Menyembunyikan** tanda audit, prioritas, dan skor LaBSE dari anotator, agar keputusan mereka tidak dipengaruhi penilaian mesin.
4. Menyertakan konteks: pasangan sebelum dan sesudahnya di dokumen yang sama.
5. Membuat dua berkas Excel yang isinya identik, satu untuk setiap anotator, dengan daftar pilihan (dropdown) di kolom `keputusan` serta lembar `Panduan` yang menjelaskan setiap kategori.
6. Menyimpan berkas kunci yang menghubungkan nomor baris dengan strata dan tanda audit, untuk analisis di tahap 9.

**Masukan:** `s07_paralel_utama.csv` (dan `s07_paralel_liturgi.csv` bila disertakan).

**Keluaran yang diharapkan:**
- `s08_review_anotator_A.xlsx` dan `s08_review_anotator_B.xlsx`: lembar kerja anotator.
- `s08_kunci_sampel.csv`: kunci strata (tidak dibagikan ke anotator).

**Dipakai untuk:** diisi oleh dua anotator secara terpisah, lalu dihitung di tahap 9. Lembar ini juga menggantikan lembar `Perlu_periksa` di buku kerja audit v1.

**Cara memeriksa:** buka salah satu berkas Excel dan pastikan dropdown kategori berfungsi serta tidak ada kolom tanda audit yang terlihat.

In [ ]:
P_S08_A = DIR_OUT / "s08_review_anotator_A.xlsx"
P_S08_B = DIR_OUT / "s08_review_anotator_B.xlsx"
P_S08_KUNCI = DIR_OUT / "s08_kunci_sampel.csv"
PARAM_RV = KONFIGURASI["review"]
KATEGORI = list(PARAM_RV["kategori_keputusan"].keys())

masukan_s08 = [P_S07_UTAMA] + ([P_S07_LITURGI] if PARAM_RV["sertakan_liturgi"] else [])
t8 = Tahap(
    "s08", "Lembar review IAA",
    masukan=masukan_s08,
    keluaran=[P_S08_A, P_S08_B, P_S08_KUNCI],
    parameter={**PARAM_RV, "seed": KONFIGURASI["seed"]},
)


def tulis_lembar_review(path, tabel, label_anotator):
    from openpyxl import Workbook
    from openpyxl.styles import Alignment, Font, PatternFill
    from openpyxl.worksheet.datavalidation import DataValidation

    wb = Workbook()
    ws = wb.active
    ws.title = "Review"
    kolom = list(tabel.columns)
    ws.append(kolom)
    for baris in tabel.itertuples(index=False):
        ws.append(["" if (isinstance(v, float) and math.isnan(v)) else v for v in baris])
    lebar = {"nomor": 7, "pair_id": 18, "keputusan": 24, "catatan": 40}
    for i, nama in enumerate(kolom, start=1):
        huruf = ws.cell(row=1, column=i).column_letter
        ws.column_dimensions[huruf].width = lebar.get(nama, 45)
        ws.cell(row=1, column=i).font = Font(bold=True)
        ws.cell(row=1, column=i).fill = PatternFill("solid", fgColor="DDDDDD")
    for row in ws.iter_rows(min_row=2):
        for sel in row:
            sel.alignment = Alignment(wrap_text=True, vertical="top")
    ws.freeze_panes = "C2"
    kol_keputusan = ws.cell(row=1, column=kolom.index("keputusan") + 1).column_letter
    dv = DataValidation(type="list", formula1='"' + ",".join(KATEGORI) + '"', allow_blank=True,
                        showErrorMessage=True, errorTitle="Kategori tidak dikenal",
                        error="Pilih salah satu kategori dari daftar.")
    ws.add_data_validation(dv)
    dv.add(f"{kol_keputusan}2:{kol_keputusan}{len(tabel) + 1}")

    pd_ws = wb.create_sheet("Panduan")
    pd_ws.append(["Lembar review korpus paralel Batak Toba dan Indonesia"])
    pd_ws.append([f"Anotator: {label_anotator}"])
    pd_ws.append(["Isi kolom 'keputusan' untuk setiap baris dengan satu kategori dari dropdown."])
    pd_ws.append(["Isi 'koreksi_bt' atau 'koreksi_id' hanya bila keputusannya perbaiki_penyelarasan atau perbaiki_terjemahan."])
    pd_ws.append(["Bekerjalah sendiri. Jangan berdiskusi dengan anotator lain sebelum kedua lembar selesai."])
    pd_ws.append([])
    pd_ws.append(["Kategori", "Penjelasan"])
    for k, v in PARAM_RV["kategori_keputusan"].items():
        pd_ws.append([k, v])
    pd_ws.column_dimensions["A"].width = 26
    pd_ws.column_dimensions["B"].width = 110
    for row in pd_ws.iter_rows():
        for sel in row:
            sel.alignment = Alignment(wrap_text=True, vertical="top")

    info = wb.create_sheet("Info")
    info.append(["versi_pipeline", KONFIGURASI["versi_pipeline"]])
    info.append(["dibuat", datetime.now(timezone.utc).isoformat(timespec="seconds")])
    info.append(["anotator", label_anotator])
    info.append(["jumlah_baris", len(tabel)])
    wb.save(path)


if not t8.bisa_dilewati():
    basis = pd.concat([baca_csv(p) for p in masukan_s08], ignore_index=True)
    basis = basis.sort_values(["doc_id", "urutan"]).reset_index(drop=True)

    konteks = {}
    for _, g in basis.groupby("doc_id"):
        g = g.reset_index(drop=True)
        for k, r in g.iterrows():
            sebelum = g.iloc[k - 1] if k > 0 else None
            sesudah = g.iloc[k + 1] if k + 1 < len(g) else None
            konteks[r["pair_id"]] = {
                "sebelum_bt": sebelum["teks_bt"] if sebelum is not None else "",
                "sebelum_id": sebelum["teks_id"] if sebelum is not None else "",
                "sesudah_bt": sesudah["teks_bt"] if sesudah is not None else "",
                "sesudah_id": sesudah["teks_id"] if sesudah is not None else "",
            }

    wajib = basis[basis["prioritas_review"].isin(["tinggi", "sedang"])]
    rutin = basis[basis["prioritas_review"] == "rutin"]
    n_rutin = min(len(rutin), max(PARAM_RV["minimal_sampel_rutin"], math.ceil(PARAM_RV["proporsi_sampel_rutin"] * len(rutin))))
    sampel_rutin = rutin.sample(n=n_rutin, random_state=KONFIGURASI["seed"])
    sampel = pd.concat([wajib, sampel_rutin]).sample(frac=1.0, random_state=KONFIGURASI["seed"]).reset_index(drop=True)
    sampel.insert(0, "nomor", range(1, len(sampel) + 1))

    lembar = pd.DataFrame({
        "nomor": sampel["nomor"],
        "pair_id": sampel["pair_id"],
        "teks_bt": sampel["teks_bt"],
        "teks_id": sampel["teks_id"],
        "keputusan": "",
        "koreksi_bt": "",
        "koreksi_id": "",
        "catatan": "",
        "konteks_sebelum_bt": sampel["pair_id"].map(lambda p: konteks[p]["sebelum_bt"]),
        "konteks_sebelum_id": sampel["pair_id"].map(lambda p: konteks[p]["sebelum_id"]),
        "konteks_sesudah_bt": sampel["pair_id"].map(lambda p: konteks[p]["sesudah_bt"]),
        "konteks_sesudah_id": sampel["pair_id"].map(lambda p: konteks[p]["sesudah_id"]),
    })
    tulis_lembar_review(P_S08_A, lembar, "A")
    tulis_lembar_review(P_S08_B, lembar, "B")
    kunci = sampel[["nomor", "pair_id", "doc_id", "korpus", "genre", "subgenre", "tipe", "metode",
                    "sim_labse", "flag_audit", "prioritas_review"]]
    simpan_csv(kunci, P_S08_KUNCI)
    t8.selesai({
        "baris_review": len(sampel),
        "prioritas_tinggi": int((sampel["prioritas_review"] == "tinggi").sum()),
        "prioritas_sedang": int((sampel["prioritas_review"] == "sedang").sum()),
        "sampel_rutin": int(n_rutin),
        "populasi_rutin": int(len(rutin)),
    })

kunci_s08 = baca_csv(P_S08_KUNCI)
print(kunci_s08["prioritas_review"].value_counts())
print("\nBagikan kedua berkas berikut kepada anotator:")
print("  ", P_S08_A)
print("  ", P_S08_B)
print("Setelah terisi, simpan ke:", DIR_REVIEW / "review_anotator_A.xlsx", "dan", DIR_REVIEW / "review_anotator_B.xlsx")

---
# Tahap 9. Kesepakatan antaranotator (IAA) dan lembar adjudikasi

**Tujuan:** mengukur seberapa konsisten dua anotator menilai pasangan yang sama, lalu menyiapkan bahan untuk menyelesaikan perbedaan.

**Tahap ini hanya berjalan bila kedua lembar yang sudah diisi tersedia** di `review_terisi/review_anotator_A.xlsx` dan `review_terisi/review_anotator_B.xlsx`. Bila belum ada, tahap ini dilewati tanpa galat.

**Yang dikerjakan:**
1. Membaca kedua lembar, memastikan `pair_id` dan urutannya sama, serta memeriksa bahwa semua keputusan berasal dari daftar kategori.
2. Menghitung untuk baris yang sudah diisi kedua anotator:
   - **kesepakatan persentase** (proporsi keputusan yang sama),
   - **Cohen's kappa** untuk semua kategori,
   - **Cohen's kappa biner**: `terima` dibandingkan dengan selain `terima`,
   - **interval kepercayaan 95%** dengan bootstrap (1.000 kali, seed tetap).
3. Memecah hasil per strata prioritas (tinggi, sedang, rutin) dan per genre.
4. Menyusun matriks konfusi antara keputusan A dan B.
5. Menyusun lembar adjudikasi: semua baris yang keputusannya berbeda, ditambah baris yang salah satu anotatornya memilih `perlu_sumber`.

**Interpretasi kappa** (Landis dan Koch, 1977): di bawah 0 buruk, 0,00 sampai 0,20 sangat lemah, 0,21 sampai 0,40 lemah, 0,41 sampai 0,60 sedang, 0,61 sampai 0,80 kuat, 0,81 sampai 1,00 hampir sempurna.

**Masukan:** dua lembar terisi dan `s08_kunci_sampel.csv`.

**Keluaran yang diharapkan:**
- `s09_laporan_iaa.json`: semua angka kesepakatan beserta interval kepercayaannya.
- `s09_matriks_konfusi.csv`.
- `s09_adjudikasi.xlsx`: baris yang perlu diputuskan bersama, dengan kolom `keputusan_akhir` dan `catatan_adjudikasi`.

**Dipakai untuk:** bagian validasi data di paper, dan sebagai dasar korpus paralel v3 yang tervalidasi.

In [ ]:
P_REV_A = DIR_REVIEW / "review_anotator_A.xlsx"
P_REV_B = DIR_REVIEW / "review_anotator_B.xlsx"
P_S09_JSON = DIR_OUT / "s09_laporan_iaa.json"
P_S09_KONF = DIR_OUT / "s09_matriks_konfusi.csv"
P_S09_ADJ = DIR_OUT / "s09_adjudikasi.xlsx"


def kappa_cohen(a, b):
    a, b = np.asarray(a), np.asarray(b)
    if len(a) == 0:
        return float("nan")
    kategori = sorted(set(a) | set(b))
    po = float(np.mean(a == b))
    pe = sum(float(np.mean(a == k)) * float(np.mean(b == k)) for k in kategori)
    return 1.0 if pe == 1 else (po - pe) / (1 - pe)


def interval_bootstrap(a, b, fungsi, n, seed):
    a, b = np.asarray(a), np.asarray(b)
    if len(a) < 2:
        return [float("nan"), float("nan")]
    rng = np.random.default_rng(seed)
    nilai = []
    for _ in range(n):
        idx = rng.integers(0, len(a), len(a))
        nilai.append(fungsi(a[idx], b[idx]))
    nilai = np.array([v for v in nilai if not np.isnan(v)])
    return [round(float(np.percentile(nilai, 2.5)), 4), round(float(np.percentile(nilai, 97.5)), 4)]


def tafsir_kappa(k):
    if np.isnan(k):
        return "tidak terdefinisi"
    for batas, label in ((0, "buruk"), (0.20, "sangat lemah"), (0.40, "lemah"), (0.60, "sedang"), (0.80, "kuat")):
        if k <= batas:
            return label
    return "hampir sempurna"


def ringkas_kesepakatan(df):
    a, b = df["keputusan_A"].to_numpy(), df["keputusan_B"].to_numpy()
    biner = lambda x: np.where(x == "terima", "terima", "bukan_terima")
    k = kappa_cohen(a, b)
    kb = kappa_cohen(biner(a), biner(b))
    return {
        "n": int(len(df)),
        "kesepakatan_persen": round(100 * float(np.mean(a == b)), 2) if len(df) else float("nan"),
        "kappa": round(k, 4),
        "kappa_tafsir": tafsir_kappa(k),
        "kappa_biner_terima": round(kb, 4),
        "kappa_biner_tafsir": tafsir_kappa(kb),
    }


if not (P_REV_A.exists() and P_REV_B.exists()):
    STATUS_TAHAP["s09"] = "menunggu lembar terisi"
    print("Tahap 9 dilewati: lembar terisi belum tersedia.")
    print("Simpan lembar yang sudah diisi ke:")
    print("  ", P_REV_A)
    print("  ", P_REV_B)
else:
    t9 = Tahap(
        "s09", "IAA dan adjudikasi",
        masukan=[P_REV_A, P_REV_B, P_S08_KUNCI],
        keluaran=[P_S09_JSON, P_S09_KONF, P_S09_ADJ],
        parameter={"kategori": KATEGORI, "bootstrap": PARAM_RV["bootstrap_iaa"], "seed": KONFIGURASI["seed"]},
    )
    if not t9.bisa_dilewati():
        def baca_lembar(path):
            df = pd.read_excel(path, sheet_name="Review", dtype=str).fillna("")
            df["keputusan"] = df["keputusan"].str.strip()
            asing = sorted(set(df["keputusan"]) - set(KATEGORI) - {""})
            if asing:
                raise ValueError(f"{path.name} berisi kategori tidak dikenal: {asing}")
            return df

        A, B = baca_lembar(P_REV_A), baca_lembar(P_REV_B)
        if A["pair_id"].tolist() != B["pair_id"].tolist():
            raise ValueError("Urutan atau isi pair_id di kedua lembar berbeda. Pastikan anotator tidak menghapus atau mengurutkan ulang baris.")
        kunci = baca_csv(P_S08_KUNCI)
        if sorted(A["pair_id"]) != sorted(kunci["pair_id"]):
            raise ValueError(
                "Lembar terisi tidak cocok dengan sampel s08 yang sekarang. Kemungkinan tahap 6 sampai 8 "
                "sudah dijalankan ulang setelah lembar dibagikan. Gunakan lembar dari sampel yang sama, "
                "atau bagikan ulang lembar s08 terbaru kepada anotator."
            )
        gabung = A[["nomor", "pair_id", "teks_bt", "teks_id", "keputusan", "koreksi_bt", "koreksi_id", "catatan"]].merge(
            B[["pair_id", "keputusan", "koreksi_bt", "koreksi_id", "catatan"]], on="pair_id", suffixes=("_A", "_B"))
        gabung = gabung.merge(kunci[["pair_id", "genre", "prioritas_review", "flag_audit"]], on="pair_id", how="left")
        lengkap = gabung[(gabung["keputusan_A"] != "") & (gabung["keputusan_B"] != "")]

        laporan = {
            "baris_total": int(len(gabung)),
            "terisi_A": int((gabung["keputusan_A"] != "").sum()),
            "terisi_B": int((gabung["keputusan_B"] != "").sum()),
            "terisi_keduanya": int(len(lengkap)),
            "keseluruhan": ringkas_kesepakatan(lengkap),
        }
        a, b = lengkap["keputusan_A"].to_numpy(), lengkap["keputusan_B"].to_numpy()
        n_boot, seed = PARAM_RV["bootstrap_iaa"], KONFIGURASI["seed"]
        laporan["keseluruhan"]["kappa_ik95"] = interval_bootstrap(a, b, kappa_cohen, n_boot, seed)
        biner = lambda x: np.where(x == "terima", "terima", "bukan_terima")
        laporan["keseluruhan"]["kappa_biner_ik95"] = interval_bootstrap(biner(a), biner(b), kappa_cohen, n_boot, seed)
        laporan["per_prioritas"] = {k: ringkas_kesepakatan(g) for k, g in lengkap.groupby("prioritas_review")}
        laporan["per_genre"] = {k: ringkas_kesepakatan(g) for k, g in lengkap.groupby("genre")}
        laporan["sebaran_keputusan_A"] = lengkap["keputusan_A"].value_counts().to_dict()
        laporan["sebaran_keputusan_B"] = lengkap["keputusan_B"].value_counts().to_dict()

        konfusi = pd.crosstab(lengkap["keputusan_A"], lengkap["keputusan_B"]).reindex(index=KATEGORI, columns=KATEGORI, fill_value=0)
        konfusi.to_csv(P_S09_KONF, encoding="utf-8")

        adj = lengkap[(lengkap["keputusan_A"] != lengkap["keputusan_B"]) |
                      (lengkap["keputusan_A"] == "perlu_sumber") | (lengkap["keputusan_B"] == "perlu_sumber")].copy()
        adj["keputusan_akhir"] = ""
        adj["teks_bt_akhir"] = ""
        adj["teks_id_akhir"] = ""
        adj["catatan_adjudikasi"] = ""
        adj = adj.drop(columns=["flag_audit"])
        tulis_lembar_review(P_S09_ADJ, adj.rename(columns={"keputusan_akhir": "keputusan"}), "adjudikasi")
        laporan["baris_adjudikasi"] = int(len(adj))

        P_S09_JSON.write_text(json.dumps(laporan, indent=2, ensure_ascii=False), encoding="utf-8")
        t9.selesai({
            "terisi_keduanya": laporan["terisi_keduanya"],
            "kappa": laporan["keseluruhan"]["kappa"],
            "kappa_ik95": laporan["keseluruhan"]["kappa_ik95"],
            "baris_adjudikasi": laporan["baris_adjudikasi"],
        })
    print(json.dumps(json.loads(P_S09_JSON.read_text(encoding="utf-8"))["keseluruhan"], indent=2, ensure_ascii=False))

---
# Tahap 10. Paket rilis, perbandingan dengan v1, dan manifest

**Tujuan:** mengumpulkan semua berkas final ke satu folder yang siap disalin ke repositori GitHub, serta mendokumentasikan perubahan angka dibanding versi sebelumnya.

**Yang dikerjakan:**
1. Menyalin berkas final ke `rilis/` dengan nama dan struktur folder yang sama dengan repositori:
   - `rilis/data/btb_multigenre.csv`
   - `rilis/data/experimental/btb_parallel_sentence_v2.csv`
   - `rilis/data/experimental/btb_parallel_liturgy_v2.csv`
   - `rilis/data/experimental/btb_unaligned_units_v2.csv`
   - `rilis/statistik/` (statistik korpus dan audit)
   - `rilis/iaa/` (bila tahap 9 sudah berjalan)
2. Bila `input/btb_parallel_sentence_v1.csv` (korpus paralel v1 dari `data/experimental/` di repositori) tersedia, menghitung berapa pasangan v1 yang muncul persis sama di v2, berapa yang hanya sisi Tobanya sama, dan berapa yang tidak ditemukan.
3. Menulis `manifest.json`: waktu, versi Python dan pustaka, konfigurasi lengkap, sidik SHA-256 input, serta sidik dan jumlah baris setiap berkas rilis.
4. Menulis `CHANGELOG_v2.md` yang merangkum perubahan metodologis dan perubahan angka.

**Masukan:** keluaran tahap 3, 5, 7, 8, dan 9.

**Keluaran yang diharapkan:** folder `rilis/` lengkap dengan `manifest.json` dan `CHANGELOG_v2.md`.

**Dipakai untuk:** pembaruan repositori GitHub (`data/` dan `data/experimental/`), README, laporan teknis, dan lampiran reproduktibilitas paper.

In [ ]:
P_V1 = DIR_INPUT / KONFIGURASI["berkas_pembanding_v1"]
salinan = {
    P_S03_RILIS: DIR_RILIS / "data" / "btb_multigenre.csv",
    P_S07_UTAMA: DIR_RILIS / "data" / "experimental" / "btb_parallel_sentence_v2.csv",
    P_S07_LITURGI: DIR_RILIS / "data" / "experimental" / "btb_parallel_liturgy_v2.csv",
    P_S07_TAK: DIR_RILIS / "data" / "experimental" / "btb_unaligned_units_v2.csv",
    P_S05_JSON: DIR_RILIS / "statistik" / "statistik_korpus.json",
    P_S05_GENRE: DIR_RILIS / "statistik" / "statistik_per_genre.csv",
    P_S05_MD: DIR_RILIS / "statistik" / "tabel_readme.md",
    P_S07_JSON: DIR_RILIS / "statistik" / "ringkasan_audit.json",
    P_S02_LOG: DIR_RILIS / "log" / "log_koreksi_metadata.csv",
    P_S03_LOG: DIR_RILIS / "log" / "log_pembersihan.csv",
    P_S08_A: DIR_RILIS / "review" / "review_anotator_A_kosong.xlsx",
    P_S08_B: DIR_RILIS / "review" / "review_anotator_B_kosong.xlsx",
}
if P_S09_JSON.exists():
    salinan[P_S09_JSON] = DIR_RILIS / "iaa" / "laporan_iaa.json"
    salinan[P_S09_KONF] = DIR_RILIS / "iaa" / "matriks_konfusi.csv"

for asal, tujuan in salinan.items():
    tujuan.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(asal, tujuan)

perbandingan_v1 = None
if P_V1.exists():
    v1 = baca_csv(P_V1)
    v2 = baca_csv(P_S07_UTAMA)
    kunci_v2 = set(zip(v2["teks_bt_norm"], v2["teks_id_norm"]))
    bt_v2 = set(v2["teks_bt_norm"])
    v1_bt = v1["toba_final"].map(normalkan_kalimat)
    v1_id = v1["indo_final"].map(normalkan_kalimat)
    sama_persis = [(b, i) in kunci_v2 for b, i in zip(v1_bt, v1_id)]
    sama_bt = [b in bt_v2 for b in v1_bt]
    perbandingan_v1 = {
        "pasangan_v1": int(len(v1)),
        "pasangan_v2_utama": int(len(v2)),
        "v1_sama_persis_di_v2": int(sum(sama_persis)),
        "v1_hanya_sisi_bt_sama": int(sum(b and not p for b, p in zip(sama_bt, sama_persis))),
        "v1_tidak_ditemukan": int(sum(not b for b in sama_bt)),
    }
    print("Perbandingan dengan v1:", perbandingan_v1)
else:
    print(f"Berkas pembanding v1 tidak ada di {P_V1}. Perbandingan dilewati.")


def versi_pustaka(nama):
    try:
        from importlib.metadata import version
        return version(nama)
    except Exception:
        return "tidak terpasang"


statistik = json.loads(P_S05_JSON.read_text(encoding="utf-8"))
audit = json.loads(P_S07_JSON.read_text(encoding="utf-8"))
manifest = {
    "nama": "Korpus Multi-genre Batak Toba dan Indonesia",
    "versi_pipeline": KONFIGURASI["versi_pipeline"],
    "dibuat": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "lingkungan": {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "colab": DI_COLAB,
        "pustaka": {p: versi_pustaka(p) for p in ("pandas", "numpy", "torch", "sentence-transformers", "nltk", "openpyxl")},
    },
    "input": {BERKAS_ANOTATOR.name: sha256_berkas(BERKAS_ANOTATOR)},
    "konfigurasi": KONFIGURASI,
    "status_tahap": STATUS_TAHAP,
    "berkas_rilis": {
        str(t.relative_to(DIR_RILIS)): {
            "sha256": sha256_berkas(t),
            "baris": int(len(baca_csv(t))) if t.suffix == ".csv" else None,
        }
        for t in salinan.values()
    },
    "perbandingan_v1": perbandingan_v1,
}
(DIR_RILIS / "manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False, default=str), encoding="utf-8")


def fmt(n):
    return f"{n:,}".replace(",", ".")


tot = statistik["total"]
u, l = audit["utama"], audit["liturgi"]
baris_cl = [
    "# CHANGELOG korpus multi-genre v2",
    "",
    f"Dibuat otomatis oleh pipeline v{KONFIGURASI['versi_pipeline']} pada {manifest['dibuat']}.",
    "",
    "## Perubahan metodologis dibanding v1",
    "",
    "1. Seluruh pengolahan berangkat dari satu berkas input (`Combined Annotator.xlsx`) dan dijalankan dalam satu notebook.",
    "2. `doc_id` mengikuti nomor dokumen anotator (`mg_NNN`), tidak lagi bergantung pada urutan berkas antara.",
    "3. Dua belas label genre atau sub-genre yang tidak sesuai panduan anotator diperbaiki, metadata contoh dari templat dikosongkan, dan semuanya dicatat di `log/log_koreksi_metadata.csv`.",
    "4. Jumlah token dilaporkan dengan tiga metode yang dinyatakan eksplisit (kata, token berbasis spasi, token NLTK v1).",
    "5. Strategi aturan 1:1 hanya dipakai bila jumlah kalimat sama. Bila tidak, dokumen dialihkan ke LaBSE (`labse_alih`).",
    "6. Blok tanpa pasangan disimpan di `btb_unaligned_units_v2.csv`, tidak dibuang.",
    "7. Agenda HKBP diselaraskan per bagian sebagai korpus liturgi terpisah.",
    "8. Audit otomatis menggunakan aturan tertulis yang berlaku untuk semua pasangan.",
    "9. Lembar review disusun untuk dua anotator dengan sampel bertingkat agar IAA bisa dihitung.",
    "",
    "## Angka v2",
    "",
    f"- Dokumen: {fmt(statistik['jumlah_dokumen'])}",
    f"- Kata Batak Toba (tanpa tanda baca): {fmt(tot['kata_bt'])}",
    f"- Token Batak Toba berbasis spasi: {fmt(tot['token_spasi_bt'])}",
    f"- Token Batak Toba NLTK termasuk tanda baca (metode v1): {fmt(tot['token_nltk_v1_bt'])}",
    f"- Pasangan korpus paralel utama: {fmt(u['pasangan'])} (perlu review: {fmt(u['status'].get('perlu_review', 0))})",
    f"- Pasangan korpus paralel liturgi: {fmt(l['pasangan'])} (perlu review: {fmt(l['status'].get('perlu_review', 0))})",
    f"- Blok tanpa pasangan: {fmt(audit['tak_berpasangan']['blok'])}",
]
if perbandingan_v1:
    baris_cl += [
        "",
        "## Perbandingan dengan korpus paralel v1",
        "",
        f"- Pasangan v1: {fmt(perbandingan_v1['pasangan_v1'])}; pasangan v2 (utama): {fmt(perbandingan_v1['pasangan_v2_utama'])}",
        f"- Pasangan v1 yang muncul persis sama di v2: {fmt(perbandingan_v1['v1_sama_persis_di_v2'])}",
        f"- Pasangan v1 yang hanya sisi Tobanya sama: {fmt(perbandingan_v1['v1_hanya_sisi_bt_sama'])}",
        f"- Pasangan v1 yang sisi Tobanya tidak ditemukan di v2: {fmt(perbandingan_v1['v1_tidak_ditemukan'])}",
    ]
if P_S09_JSON.exists():
    iaa = json.loads(P_S09_JSON.read_text(encoding="utf-8"))["keseluruhan"]
    baris_cl += ["", "## IAA", "",
                 f"- n = {iaa['n']}, kesepakatan {iaa['kesepakatan_persen']}%, kappa {iaa['kappa']} ({iaa['kappa_tafsir']}), IK95% {iaa.get('kappa_ik95')}"]
(DIR_RILIS / "CHANGELOG_v2.md").write_text("\n".join(baris_cl) + "\n", encoding="utf-8")
STATUS_TAHAP["s10"] = "dijalankan"
print("Rilis tersimpan di", DIR_RILIS)
for p in sorted(DIR_RILIS.rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(DIR_RILIS))

---
# Ringkasan akhir

Sel ini menampilkan status setiap tahap pada eksekusi terakhir (`dijalankan`, `checkpoint`, atau `menunggu lembar terisi`) dan angka kunci dari setiap tahap. Salin tabel ini ke catatan penelitian setiap kali pipeline dijalankan ulang.

In [ ]:
baris = []
for kode in ["s01", "s02", "s03", "s04", "s05", "s06", "s07", "s08", "s09", "s10"]:
    meta_path = DIR_META / f"{kode}.json"
    meta = json.loads(meta_path.read_text(encoding="utf-8")) if meta_path.exists() else {}
    baris.append({
        "tahap": kode,
        "nama": meta.get("nama", "Paket rilis" if kode == "s10" else ""),
        "status": STATUS_TAHAP.get(kode, "belum dijalankan"),
        "waktu": meta.get("waktu", ""),
        "ringkasan": json.dumps(meta.get("ringkasan", {}), ensure_ascii=False),
    })
ringkasan_akhir = pd.DataFrame(baris)
tampilkan(ringkasan_akhir, 20)